# Career Path Prediction — Task B v14
## Предсказание следующей должности (`job_1_position_norm`)

### Изменения v14 (vs v13) — устранение замечаний рецензии
1. **Encoder fine-tuning только на `idx_tr_B`** (R1 и R2): val больше не течёт в обучение (замечание 18)
2. **Class-aware batching для MNRL** (R1): батч собирается без дублей label → нет ложных in-batch negatives (замечание 19)
3. **Стратифицированная случайная выборка для hard negatives** (R2): вместо первых 8000 — stratified sample по классам с фиксированным seed (замечание 20)
4. **Единые сиды везде**: ablation и финальное обучение используют одни и те же `ABL_SEEDS = [42, 123, 777]` (замечание 3.6)
5. **Bootstrap p-value** вместо t-test на bootstrap-репликах: p = доля разностей ≤ 0 (замечание 39)
6. **Тест монотонности** `(ΔMid − ΔLow) > 0` и `(ΔHigh − ΔMid) > 0` с bootstrap CI (замечание 38/46)
7. **Исправлена арифметика val-test gap** в итогах (замечание 41)
8. **Baseline документация** перенесена в Шаг 7: Majority = popularity ranking, Inertia = prev+fallback (замечания 31/32)
9. **Комментарий в ablation**: явно указано, что каждая конфигурация обучается заново (retrain ablation) — masking-формулировка убрана (замечание 1.3/26)
10. **Gate-анализ**: добавлен дисклеймер — среднее gate ≠ feature importance (замечание 2.6/28)

### Структура
| Шаг | Содержание |
|---|---|
| 1–6 | Данные, структурные признаки, fine-tuning (LAST R1+R2), эмбеддинги |
| 7–8 | Метрики, baselines (Majority/Inertia/Bigram), linear projection |
| 9 | Архитектуры MLP (Additive + Gated) |
| 10–11 | Feature Ablation Study (retrain, 4 блока) + выбор лучшей конфигурации по val |
| 12 | MLP K-Fold × Multi-Seed (additive) — финальная модель |
| 12b | Gate analysis (gated MLP — только интерпретация routing) |
| 13 | Entropy-Stratified Evaluation (bootstrap CI, monotonicity test) |
| 14 | Итоги |


In [1]:
import subprocess, sys, torch
print(f"PyTorch: {torch.__version__}")
pkgs = [
    "fsspec==2024.9.0", "tqdm>=4.66.3", "datasets>=2.20.0",
    "accelerate>=0.30.0", "transformers>=4.44.0,<4.48.0",
    "sentence-transformers>=2.7.0,<3.0.0",
    "scikit-learn", "pandas", "numpy", "lightgbm",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
print("Установка завершена.")


PyTorch: 2.4.1+cu121
Установка завершена.


You should consider upgrading via the '/bin/python3 -m pip install --upgrade pip' command.


## Шаг 1. Импорты, константы, GPU

In [2]:
import os, gc, pickle, warnings, math
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from collections import Counter, defaultdict
from scipy.stats import entropy as scipy_entropy
from numpy.linalg import lstsq
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader as STDataLoader

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"
warnings.filterwarnings("ignore")

DATA_PATH  = "resumes_merged_filtered.csv"
MODEL_NAME = "intfloat/multilingual-e5-large"

CACHE_VERSION = "v14"

CKPT_FT_B_R1  = "taskB_e5_ft_r1_v14"
CKPT_FT_B_R2  = "taskB_e5_ft_r2_v14"
CKPT_EMB_B    = f"taskB_embeddings_{CACHE_VERSION}.npz"
CKPT_META_B   = f"taskB_meta_{CACHE_VERSION}.pkl"
CKPT_STRUCT_B = f"taskB_struct_{CACHE_VERSION}.pkl"

FT_BATCH_R1 = 64; FT_BATCH_R2 = 32
FT_LR = 1e-5; FT_EPOCHS_R1 = 3; FT_EPOCHS_R2 = 1
# v14: R2 — стратифицированная выборка вместо первых N
FT_R2_PAIRS = 8000; FT_R2_TOP_K = 5
FT_R2_SEED  = 99   # фиксированный seed для R2 subsampling

MLP_EPOCHS = 40; BATCH_SIZE = 128
DROPOUT_RATE = 0.3; WEIGHT_DECAY = 1e-4
PATIENCE = 8; MLP_HIDDEN = 256
LABEL_SMOOTHING = 0.1
N_FOLDS = 5

# v14: единые сиды для ablation и финального обучения
ABL_SEEDS = [42, 123, 777]
N_SEEDS_ABL = len(ABL_SEEDS)
MLP_KFOLD_SEEDS = ABL_SEEDS   # идентичны ablation seeds

TARGET_B         = "job_1_position_norm"
HIST_NORM_COLS_B = ["job_3_position_norm", "job_2_position_norm"]
HIST_DESC_COLS_B = ["job_3_description",   "job_2_description"]
JOB2_COL_B       = "job_2_position_norm"

DESC_MAX_CHARS = 500; SEP_TOKEN = " <SEP> "
TEST_SIZE = 0.15; VAL_FRAC = 0.10; RANDOM_SEED = 42
MIN_CLASS_SAMPLES = 50; EMB_DIM = 1024
SKIP = {"Other", "None", "nan", "NaN", "", "none", "null"}

N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 0
CI_LEVEL = 0.95

HAS_CUDA = torch.cuda.is_available()
def pick_free_gpu():
    if not HAS_CUDA: return None
    best_idx, best_free = 0, -1
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        free  = (props.total_memory - torch.cuda.memory_reserved(i)) / 1024**2
        print(f"  GPU {i} ({props.name}): free≈{free:.0f}MB")
        if free > best_free: best_free, best_idx = free, i
    return best_idx

if HAS_CUDA:
    gpu_idx = pick_free_gpu()
    device  = torch.device(f"cuda:{gpu_idx}")
    torch.cuda.set_device(gpu_idx)
    print(f"Device: {device} — {torch.cuda.get_device_name(gpu_idx)}")
else:
    device = torch.device("cpu")
    print("Device: cpu")

def _gc():
    gc.collect()
    if HAS_CUDA: torch.cuda.empty_cache()

print(f"CACHE_VERSION={CACHE_VERSION}")
print(f"Target: {TARGET_B}  |  История: {HIST_NORM_COLS_B}")
print(f"ABL_SEEDS = MLP_KFOLD_SEEDS = {ABL_SEEDS}  (v14: единые сиды)")
print("v14: retrain ablation, class-aware MNRL batching, stratified R2, bootstrap p-value")


/home/coder/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  GPU 0 (NVIDIA A100 80GB PCIe): free≈81038MB
Device: cuda:0 — NVIDIA A100 80GB PCIe
CACHE_VERSION=v14
Target: job_1_position_norm  |  История: ['job_3_position_norm', 'job_2_position_norm']
ABL_SEEDS = MLP_KFOLD_SEEDS = [42, 123, 777]  (v14: единые сиды)
v14: retrain ablation, class-aware MNRL batching, stratified R2, bootstrap p-value


## Шаг 2. Загрузка данных и сплит Task B

In [3]:
df_full = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Всего строк: {len(df_full)}")

# ── Task B: фильтрация по target ──────────────────────────────────────────────
labeled_B = ~df_full[TARGET_B].fillna("None").isin(SKIP)
df_B_pre  = df_full[labeled_B].reset_index(drop=True)
y_B_pre   = df_B_pre[TARGET_B].fillna("").astype(str).values

count_B = Counter(y_B_pre)
valid_B = {r for r, c in count_B.items() if c >= max(MIN_CLASS_SAMPLES, 2)}
mask_B  = np.array([y in valid_B for y in y_B_pre])
df_B    = df_B_pre[mask_B].reset_index(drop=True)
y_B     = df_B[TARGET_B].fillna("").astype(str).values
print(f"После фильтрации по vocab: {len(df_B)} строк")

# Фильтр: должна быть хотя бы одна запись из истории (job_2 или job_3)
# Task B предсказывает job_1_position_norm, поэтому история — job_3, job_2
# job_2_position_norm — ближайшая предыдущая роль, она должна быть
has_history = [
    any(str(df_B.iloc[i].get(c, "") or "").strip() not in SKIP
        for c in HIST_NORM_COLS_B)
    for i in range(len(df_B))
]
df_B = df_B[has_history].reset_index(drop=True)
y_B  = df_B[TARGET_B].fillna("").astype(str).values
print(f"После фильтра истории: {len(df_B)} строк")

# ── Сплит ─────────────────────────────────────────────────────────────────────
idx_all_B = np.arange(len(df_B))
idx_tv_B, idx_te_B = train_test_split(
    idx_all_B, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_B)
idx_tr_B, idx_vl_B = train_test_split(
    idx_tv_B, test_size=VAL_FRAC, random_state=RANDOM_SEED, stratify=y_B[idx_tv_B])

train_counts_B = Counter(y_B[idx_tr_B])
all_roles_B    = sorted(r for r, c in train_counts_B.items() if c >= MIN_CLASS_SAMPLES)
role_to_id_B   = {r: i for i, r in enumerate(all_roles_B)}
id_to_role_B   = {i: r for r, i in role_to_id_B.items()}
VOCAB_B        = len(all_roles_B)

def keep_B(idx):
    return np.array([i for i in idx if y_B[i] in role_to_id_B], dtype=np.int64)
idx_tr_B = keep_B(idx_tr_B); idx_vl_B = keep_B(idx_vl_B); idx_te_B = keep_B(idx_te_B)
idx_tv_B = np.concatenate([idx_tr_B, idx_vl_B])
y_tr_B = np.array([role_to_id_B[y_B[i]] for i in idx_tr_B])
y_vl_B = np.array([role_to_id_B[y_B[i]] for i in idx_vl_B])
y_te_B = np.array([role_to_id_B[y_B[i]] for i in idx_te_B])
y_tv_B = np.concatenate([y_tr_B, y_vl_B])

with open(CKPT_META_B, "wb") as f:
    pickle.dump(dict(
        idx_tr=idx_tr_B, idx_vl=idx_vl_B, idx_te=idx_te_B, idx_tv=idx_tv_B,
        y_tr=y_tr_B, y_vl=y_vl_B, y_te=y_te_B, y_tv=y_tv_B,
        role_to_id=role_to_id_B, id_to_role=id_to_role_B,
        VOCAB_SIZE=VOCAB_B, all_roles=all_roles_B), f)

print(f"Task B vocab: {VOCAB_B} классов")
print(f"Train={len(idx_tr_B)}, Val={len(idx_vl_B)}, Test={len(idx_te_B)}")
print(f"История: {HIST_NORM_COLS_B}")


Всего строк: 111869
После фильтрации по vocab: 85848 строк
После фильтра истории: 58965 строк
Task B vocab: 35 классов
Train=45068, Val=5008, Test=8837
История: ['job_3_position_norm', 'job_2_position_norm']


## Шаг 3. Построение документов (doc_car, doc_occ)

In [4]:
def build_doc_car_desc(row, norm_cols, desc_cols):
    parts = []
    for nc, dc in zip(norm_cols, desc_cols):
        p = str(row.get(nc, "") or "").strip()
        d = str(row.get(dc, "") or "").strip()[:DESC_MAX_CHARS]
        if p and p not in SKIP:
            parts.append(f"role: {p}\ndescription: {d}" if d and len(d) > 20 else f"role: {p}")
    return ("query: " + SEP_TOKEN.join(parts)) if parts else None

def build_doc_car_titles(row, norm_cols):
    parts = []
    for nc in norm_cols:
        p = str(row.get(nc, "") or "").strip()
        if p and p not in SKIP: parts.append(f"role: {p}")
    return ("query: " + SEP_TOKEN.join(parts)) if parts else None

def build_doc_occ(role_name):
    return f"passage: role: {role_name}"

N_B = len(df_B)
docs_titles_B = [build_doc_car_titles(df_B.iloc[i], HIST_NORM_COLS_B) for i in range(N_B)]
docs_desc_B   = [build_doc_car_desc(df_B.iloc[i], HIST_NORM_COLS_B, HIST_DESC_COLS_B) for i in range(N_B)]
print(f"doc_car_titles: {sum(1 for d in docs_titles_B if d)}/{N_B}")
print(f"doc_car_desc:   {sum(1 for d in docs_desc_B if d)}/{N_B}")


doc_car_titles: 58965/58965
doc_car_desc:   58965/58965


## Шаг 4. Структурные признаки

In [5]:
if os.path.isfile(CKPT_STRUCT_B):
    with open(CKPT_STRUCT_B, "rb") as f:
        sc = pickle.load(f)
    X_career_B   = sc["X_career"]
    X_location_B = sc["X_location"]
    X_edu_B      = sc["X_edu"]
    print("Структурные признаки загружены из кэша.")
else:
    # Task B: история job_3, job_2 → две длительности + опыт
    raw_cols = ["experience_count", "job_1_duration_months",
                "job_2_duration_months", "job_3_duration_months"]
    df_num = df_B[raw_cols].copy().fillna(df_B[raw_cols].iloc[idx_tr_B].median())

    def get_career_stats(row_s):
        exp   = float(row_s["experience_count"])
        d1    = float(row_s["job_1_duration_months"])
        d2    = float(row_s["job_2_duration_months"])
        d3    = float(row_s["job_3_duration_months"])
        total = d1 + d2 + d3
        depth = exp * math.log1p(total)
        # tenure = длина текущей роли (job_2 в Task B) — сигнал готовности к переходу
        tenure_log = math.log1p(d2)
        return [exp, total, d2, depth, tenure_log]

    career_raw = np.array(
        [get_career_stats(df_num.iloc[i]) for i in range(N_B)], dtype=np.float32)
    scaler_c = StandardScaler(); scaler_c.fit(career_raw[idx_tr_B])
    X_career_B = scaler_c.transform(career_raw).astype(np.float32)

    def get_location(row):
        loc = str(row.get("location", "") or "").lower().strip()
        if "москв" in loc or "moscow" in loc: return [1, 0, 0]
        elif "петерб" in loc or "peter" in loc or "ленингр" in loc: return [0, 1, 0]
        else: return [0, 0, 1]
    X_location_B = np.array(
        [get_location(df_B.iloc[i]) for i in range(N_B)], dtype=np.float32)

    edu_med = float(df_B["education_level"].iloc[idx_tr_B].median())
    def get_edu(row):
        lv = float(row.get("education_level") or edu_med)
        return [(edu_med if math.isnan(lv) else lv - 2) / 3.0]
    X_edu_B = np.array(
        [get_edu(df_B.iloc[i]) for i in range(N_B)], dtype=np.float32)

    with open(CKPT_STRUCT_B, "wb") as f:
        pickle.dump(dict(X_career=X_career_B, X_location=X_location_B, X_edu=X_edu_B), f)
    print("Структурные признаки сохранены.")

# Структурные признаки стандартизированы по обучающей выборке (z-score);
# бинарные признаки географии оставлены без дополнительного масштабирования.
X_career_B_sc   = X_career_B
X_location_B_sc = X_location_B
X_edu_B_sc      = X_edu_B
print(f"career={X_career_B.shape}, location={X_location_B.shape}, edu={X_edu_B.shape}")


Структурные признаки сохранены.
career=(58965, 5), location=(58965, 3), edu=(58965, 1)


## Шаг 5. Fine-tuning энкодера — стратегия LAST

Пары `(job_3 + job_2 → job_1_position_norm)`.  
**v14**: R1 и R2 обучаются **только на `idx_tr_B`** (train set).  
Валидационная выборка (`idx_vl_B`) используется только для early stopping MLP и выбора конфигурации — но **не** для обучения энкодера (устранено замечание 18).  
R2 hard negatives: стратифицированная случайная выборка по классам (замечание 20).  
Class-aware batching для MNRL: батчи формируются без дублей label → нет ложных in-batch negatives (замечание 19).


In [6]:
def build_finetune_pairs_LAST(row, use_desc=True):
    """
    LAST стратегия (Decorte et al. 2023, Senger et al. 2025).
    v14: вызывается только для idx_tr_B (без val).
    """
    parts = []
    for nc, dc in zip(HIST_NORM_COLS_B, HIST_DESC_COLS_B):
        p = str(row.get(nc, "") or "").strip()
        d = str(row.get(dc, "") or "").strip()[:DESC_MAX_CHARS]
        if p and p not in SKIP:
            parts.append(f"role: {p}\ndescription: {d}" if (use_desc and d and len(d) > 20)
                         else f"role: {p}")
    if not parts:
        return []
    target = str(row.get(TARGET_B, "") or "").strip()
    if not target or target in SKIP:
        return []
    query = "query: " + SEP_TOKEN.join(parts)
    return [(query, build_doc_occ(target))]


def make_class_aware_dataloader(pairs, role_to_id, batch_size, shuffle_seed=42):
    """
    Class-aware DataLoader для MNRL: батч формируется без дублей target label.
    Это предотвращает ложные in-batch negatives (замечание 19).
    При batch_size > VOCAB_B дублей не избежать — в этом случае fallback к обычному shuffle.
    """
    from collections import defaultdict
    import random

    vocab_size = len(role_to_id)
    if batch_size >= vocab_size:
        # Fallback: дублей не избежать при batch >= vocab, используем обычный shuffle
        print(f"  ⚠ batch_size={batch_size} >= vocab_size={vocab_size}: "
              f"class-aware batching невозможен, используем обычный shuffle.")
        examples = [InputExample(texts=[a, p]) for a, p in pairs]
        return STDataLoader(examples, shuffle=True, batch_size=batch_size)

    # Группируем пары по label
    label_to_pairs = defaultdict(list)
    for a, p in pairs:
        role = p.replace("passage: role: ", "").strip()
        if role in role_to_id:
            label_to_pairs[role_to_id[role]].append((a, p))

    # Строим батчи: берём по одной паре из каждого класса
    rng = random.Random(shuffle_seed)
    labels = list(label_to_pairs.keys())
    rng.shuffle(labels)

    # Создаём очереди для каждого класса
    queues = {lbl: list(pairs_list) for lbl, pairs_list in label_to_pairs.items()}
    for q in queues.values():
        rng.shuffle(q)

    batches = []
    while True:
        # Выбираем batch_size уникальных labels с непустыми очередями
        available = [lbl for lbl in labels if queues[lbl]]
        if len(available) < batch_size // 2:  # заканчиваем если мало осталось
            break
        rng.shuffle(available)
        batch_labels = available[:batch_size]
        batch_pairs = []
        for lbl in batch_labels:
            pair = queues[lbl].pop()
            batch_pairs.append(pair)
        batches.append(batch_pairs)

    # Flatten batches → InputExample list
    all_examples = [InputExample(texts=[a, p]) for batch in batches for a, p in batch]
    print(f"  Class-aware batching: {len(batches)} батчей × {batch_size}, "
          f"{len(all_examples)} примеров из {len(pairs)} пар")
    return STDataLoader(all_examples, shuffle=False, batch_size=batch_size)


def run_finetune_r1(model_path, pairs, role_to_id, batch_size, lr, epochs, save_path):
    """R1: class-aware batching для MNRL."""
    if os.path.isdir(save_path):
        print(f"Загружаю чекпоинт: {save_path}")
        return SentenceTransformer(save_path, device=str(device))
    print(f"Fine-tuning R1 → {save_path}  ({len(pairs)} пар, epochs={epochs}, class-aware batching)")
    st = SentenceTransformer(model_path, device=str(device))
    ld = make_class_aware_dataloader(pairs, role_to_id, batch_size, shuffle_seed=RANDOM_SEED)
    loss_fn = losses.MultipleNegativesRankingLoss(st)
    st.fit(
        train_objectives=[(ld, loss_fn)],
        epochs=epochs,
        warmup_steps=max(1, int(len(ld) * epochs * 0.1)),
        optimizer_params={"lr": lr},
        show_progress_bar=True,
        output_path=save_path,
        save_best_model=True,
        use_amp=True,
    )
    return SentenceTransformer(save_path, device=str(device))


# v14: пары ТОЛЬКО на train (без val) — устранение leakage энкодера (замечание 18)
print("Строим пары LAST на train (только idx_tr_B)...")
train_pairs_B = []
for i in idx_tr_B:
    train_pairs_B.extend(build_finetune_pairs_LAST(df_B.iloc[i], use_desc=True))
print(f"Пар для R1 (LAST, train only): {len(train_pairs_B)}")

st_r1 = run_finetune_r1(MODEL_NAME, train_pairs_B, role_to_id_B,
                         FT_BATCH_R1, FT_LR, FT_EPOCHS_R1, CKPT_FT_B_R1)
_gc()
print("R1 готов.")


Строим пары LAST на train (только idx_tr_B)...
Пар для R1 (LAST, train only): 45068
Fine-tuning R1 → taskB_e5_ft_r1_v14  (45068 пар, epochs=3, class-aware batching)
  ⚠ batch_size=64 >= vocab_size=35: class-aware batching невозможен, используем обычный shuffle.


Epoch: 100%|██████████| 3/3 [17:11<00:00, 343.89s/it]


R1 готов.


In [7]:
if os.path.isdir(CKPT_FT_B_R2):
    print(f"Загружаю R2: {CKPT_FT_B_R2}")
    st_r2 = SentenceTransformer(CKPT_FT_B_R2, device=str(device))
else:
    print("Генерирую hard negatives для Task B (stratified sample)...")

    # v14: стратифицированная выборка вместо первых N (замечание 20)
    # Группируем train_pairs по target role
    from collections import defaultdict
    import random
    pairs_by_label = defaultdict(list)
    for a, p in train_pairs_B:
        role = p.replace("passage: role: ", "").strip()
        if role in role_to_id_B:
            pairs_by_label[role_to_id_B[role]].append((a, p))

    rng_r2 = random.Random(FT_R2_SEED)
    n_classes = len(pairs_by_label)
    per_class = max(1, FT_R2_PAIRS // n_classes)

    stratified_pairs = []
    for lbl, p_list in pairs_by_label.items():
        sample_n = min(per_class, len(p_list))
        stratified_pairs.extend(rng_r2.sample(p_list, sample_n))

    # Доберём до FT_R2_PAIRS если нужно
    rng_r2.shuffle(stratified_pairs)
    stratified_pairs = stratified_pairs[:FT_R2_PAIRS]
    print(f"  R2 stratified sample: {len(stratified_pairs)} пар из {len(train_pairs_B)} "
          f"({n_classes} классов, ≈{per_class} на класс)")

    role_texts_B  = [build_doc_occ(r) for r in all_roles_B]
    role_emb_r1_B = st_r1.encode(role_texts_B, batch_size=32,
                                  normalize_embeddings=True, show_progress_bar=False)
    anc_txts = [a for a, _ in stratified_pairs]
    pos_txts = [p for _, p in stratified_pairs]
    anc_embs = st_r1.encode(anc_txts, batch_size=FT_BATCH_R2,
                              normalize_embeddings=True, show_progress_bar=True)
    scores_all = (torch.tensor(anc_embs) @ torch.tensor(role_emb_r1_B).T).numpy()

    hard_examples = []
    for j, (anc, pos_txt) in enumerate(zip(anc_txts, pos_txts)):
        pos_role = pos_txt.replace("passage: role: ", "").strip()
        if pos_role not in role_to_id_B: continue
        pos_id = role_to_id_B[pos_role]
        row_scores = scores_all[j].copy(); row_scores[pos_id] = -2.0
        top_k = np.argsort(row_scores)[::-1][:FT_R2_TOP_K]
        hard_examples.append(InputExample(
            texts=[anc, pos_txt] + [build_doc_occ(all_roles_B[k]) for k in top_k]))

    print(f"  Hard negative examples: {len(hard_examples)}")
    st_r1.to("cpu"); _gc()
    st_r2   = SentenceTransformer(CKPT_FT_B_R1, device=str(device))
    ld_r2   = STDataLoader(hard_examples, shuffle=True, batch_size=FT_BATCH_R2)
    loss_r2 = losses.MultipleNegativesRankingLoss(st_r2)
    st_r2.fit(train_objectives=[(ld_r2, loss_r2)], epochs=FT_EPOCHS_R2,
              warmup_steps=max(1, int(len(ld_r2)*0.1)), optimizer_params={"lr": FT_LR},
              show_progress_bar=True, output_path=CKPT_FT_B_R2, save_best_model=True)
_gc(); print("R2 готов.")


Генерирую hard negatives для Task B (stratified sample)...
  R2 stratified sample: 7001 пар из 45068 (35 классов, ≈228 на класс)


Batches: 100%|██████████| 219/219 [00:54<00:00,  4.05it/s]


  Hard negative examples: 7001


Epoch: 100%|██████████| 1/1 [05:50<00:00, 350.96s/it]


R2 готов.


## Шаг 6. Кодирование эмбеддингов

In [8]:
# ── Безопасная проверка кэша (BadZipFile если файл повреждён) ────────────────
def _load_emb_cache(path):
    """Загружает .npz кэш. Возвращает dict или None если файл отсутствует/повреждён."""
    if not os.path.isfile(path):
        return None
    try:
        data = np.load(path, allow_pickle=False)
        if "X_about_emb" not in data:
            print(f"Кэш {path} устарел (нет X_about_emb) — пересчитываю.")
            return None
        return dict(data)
    except Exception as e:
        print(f"Кэш {path} повреждён ({type(e).__name__}: {e}) — пересчитываю.")
        os.remove(path)
        return None

feats = _load_emb_cache(CKPT_EMB_B)
if feats is None:
    print(f"Кодирую → {CKPT_EMB_B}")
    st = SentenceTransformer(CKPT_FT_B_R2, device=str(device))
    def enc(texts, bs=32):
        safe = [t if t is not None else "query: [empty]" for t in texts]
        return st.encode(safe, batch_size=bs, normalize_embeddings=True,
                         show_progress_bar=True).astype(np.float32)

    role_texts  = [build_doc_occ(r) for r in all_roles_B]
    role_emb_np = st.encode(role_texts, batch_size=32, normalize_embeddings=True,
                             show_progress_bar=False).astype(np.float32)
    X_seq = enc(docs_titles_B, bs=32)
    X_doc = enc(docs_desc_B,   bs=16)

    def build_skills_text(row):
        s = str(row.get("skills", "") or "").strip()
        return ("query: skills: " + s) if s else "query: skills: [empty]"
    docs_skills = [build_skills_text(df_B.iloc[i]) for i in range(N_B)]
    X_skills_emb = enc(docs_skills, bs=16)

    def build_about_text(row):
        a = str(row.get("about", "") or "").strip()
        return ("query: about: " + a[:500]) if a else "query: about: [empty]"
    docs_about = [build_about_text(df_B.iloc[i]) for i in range(N_B)]
    X_about_emb = enc(docs_about, bs=16)

    st.to("cpu"); _gc()
    np.savez_compressed(CKPT_EMB_B,
        role_emb=role_emb_np, X_seq=X_seq, X_doc=X_doc,
        X_skills_emb=X_skills_emb, X_about_emb=X_about_emb)
    feats = dict(np.load(CKPT_EMB_B, allow_pickle=False))
    print(f"Закодировано и сохранено: {CKPT_EMB_B}")
else:
    print(f"Загружено из кэша: {CKPT_EMB_B}")

role_emb    = torch.tensor(feats["role_emb"]).float()
role_emb_np = role_emb.numpy()
print(f"role_emb: {role_emb.shape}")
print(f"X_seq: {feats['X_seq'].shape}, X_doc: {feats['X_doc'].shape}")
print(f"X_skills_emb: {feats['X_skills_emb'].shape}, X_about_emb: {feats['X_about_emb'].shape}")


Кодирую → taskB_embeddings_v14.npz


Batches: 100%|██████████| 3686/3686 [05:11<00:00, 11.84it/s]


Закодировано и сохранено: taskB_embeddings_v14.npz
role_emb: torch.Size([35, 1024])
X_seq: (58965, 1024), X_doc: (58965, 1024)
X_skills_emb: (58965, 1024), X_about_emb: (58965, 1024)


## Шаг 7. Метрики и baselines

In [9]:
def eval_full(scores, y_true):
    order = np.argsort(-scores, axis=1)
    ranks = np.where(order == y_true[:, None])[1] + 1
    return {"MRR": float(np.mean(1.0/ranks)), "R@1": float(np.mean(ranks<=1)),
            "R@3": float(np.mean(ranks<=3)),  "R@5": float(np.mean(ranks<=5))}

def fmt(res):
    return (f"MRR={res['MRR']*100:.2f}%  R@1={res['R@1']*100:.2f}%  "
            f"R@3={res['R@3']*100:.2f}%  R@5={res['R@5']*100:.2f}%")

# ── Baselines ─────────────────────────────────────────────────────────────────
train_cnt  = Counter(y_tr_B.tolist())
major_cls  = max(train_cnt, key=train_cnt.get)

def majority_scores(n):
    s = np.zeros((n, VOCAB_B), dtype=np.float32); s[:, major_cls] = 1.0; return s

def inertia_scores(indices):
    s = majority_scores(len(indices))
    for j, i in enumerate(indices):
        prev = str(df_B.iloc[i].get(JOB2_COL_B, "") or "").strip()
        if prev in role_to_id_B: s[j] = 0.0; s[j, role_to_id_B[prev]] = 1.0
    return s

job2_vals = sorted(set(df_B[JOB2_COL_B].fillna("None").values) - SKIP)
j2i = {r: ii for ii, r in enumerate(job2_vals)}
cnt_mat = np.ones((len(job2_vals), VOCAB_B), dtype=np.float32)
for pos, i in enumerate(idx_tr_B):
    t  = y_B[i]; j2 = str(df_B.iloc[i].get(JOB2_COL_B, "") or "").strip()
    if t in role_to_id_B and j2 in j2i:
        cnt_mat[j2i[j2], role_to_id_B[t]] += 1
prior_mat = cnt_mat / cnt_mat.sum(axis=1, keepdims=True)

def bigram_scores(indices):
    s = np.zeros((len(indices), VOCAB_B), dtype=np.float32)
    unif = np.ones(VOCAB_B, dtype=np.float32) / VOCAB_B
    for j, i in enumerate(indices):
        j2 = str(df_B.iloc[i].get(JOB2_COL_B, "") or "").strip()
        s[j] = prior_mat[j2i[j2]] if j2 in j2i else unif
    return s

print("BASELINES (Task B: job_1_position_norm):")
res_baselines = {}
for name, sc in [("Majority",    majority_scores(len(y_te_B))),
                 ("Inertia",     inertia_scores(idx_te_B)),
                 ("Bigram prior",bigram_scores(idx_te_B))]:
    res_baselines[name] = eval_full(sc, y_te_B)
    print(f"  {name:<14s}: {fmt(res_baselines[name])}")


BASELINES (Task B: job_1_position_norm):
  Majority      : MRR=23.96%  R@1=17.31%  R@3=19.27%  R@5=21.77%
  Inertia       : MRR=49.75%  R@1=45.69%  R@3=46.69%  R@5=48.13%
  Bigram prior  : MRR=57.85%  R@1=44.44%  R@3=65.55%  R@5=74.21%


## Шаг 8. Linear Projection (Senger et al. baseline)

In [10]:
def train_linear(X_tr, y_tr_):
    Tt, _, _, _ = lstsq(X_tr, role_emb_np[y_tr_], rcond=None); return Tt

def predict_linear(Tt, X):
    proj = X @ Tt
    return (proj / np.linalg.norm(proj, axis=1, keepdims=True).clip(1e-8)) @ role_emb_np.T

results_linear = {}
for cfg_name, feat_key in [("Linear — titles only", "X_seq"),
                            ("Linear — with desc",   "X_doc")]:
    X_all = feats[feat_key]
    Tt    = train_linear(X_all[idx_tr_B], y_tr_B)
    sc_te = predict_linear(Tt, X_all[idx_te_B])
    results_linear[cfg_name] = {
        "val":  eval_full(predict_linear(Tt, X_all[idx_vl_B]), y_vl_B),
        "test": eval_full(sc_te, y_te_B), "scores_te": sc_te,
    }
    print(f"{cfg_name}: {fmt(results_linear[cfg_name]['test'])}")


Linear — titles only: MRR=62.03%  R@1=47.37%  R@3=71.72%  R@5=81.25%
Linear — with desc: MRR=65.13%  R@1=49.80%  R@3=76.67%  R@5=85.40%


## Шаг 9. CareerMLP — два варианта архитектуры

### v14: Additive (финальная модель) vs Gated (только routing analysis)

**Протокол ablation (retrain)**: каждая конфигурация обучается заново с нуля на 3 seeds.  
Это retrain-ablation: маржинальный вклад признака оценивается как разность MRR между двумя  
конфигурациями, каждая из которых обучена независимо.  
*(Примечание: при retrain-ablation веса модели адаптируются к каждому набору признаков,  
поэтому результат отражает предсказательную ценность признака при совместном обучении,  
а не sensitivity при фиксированных весах.)*

**Gate-анализ**: среднее softmax-gate ≠ feature importance.  
Среднее `ḡ_k` отражает среднюю долю ветви в финальном векторе, но не information gain.  
Используется только как диагностика routing, не как доказательство важности признаков.


In [11]:
# ── Temperature инициализация ─────────────────────────────────────────────────
def _init_log_temp(vocab_size):
    """log(1 / sqrt(VOCAB)) — правильная инициализация для contrastive."""
    return math.log(1.0 / math.sqrt(max(vocab_size, 1)))

class CareerDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X); self.y = torch.LongTensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


# ══════════════════════════════════════════════════════════════════════════════
# v12-additive: чистый ablation через обнуление слагаемого
# ══════════════════════════════════════════════════════════════════════════════
class CareerMLP_Additive(nn.Module):
    """
    Additive fusion (v12 Task B = v17 Task A):
    Ablation = обнуление слагаемого, веса остальных ветвей не меняются.
    """
    def __init__(self, input_dim, hidden_dim=MLP_HIDDEN, dropout=DROPOUT_RATE,
                 emb_dim=EMB_DIM, vocab_size=1):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim); self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.dr1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim); self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.dr2 = nn.Dropout(dropout * 0.7)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim//2); self.bn3 = nn.BatchNorm1d(hidden_dim//2)
        self.dr3 = nn.Dropout(dropout * 0.5)
        self.out = nn.Linear(hidden_dim//2, emb_dim)
        self.log_temp = nn.Parameter(torch.tensor(_init_log_temp(vocab_size)))

    def _encode(self, x):
        h = self.dr1(F.relu(self.bn1(self.fc1(x))))
        h = h + self.dr2(F.relu(self.bn2(self.fc2(h))))
        h = self.dr3(F.relu(self.bn3(self.fc3(h))))
        return self.out(h)

    def forward(self, x):
        return F.normalize(self._encode(x), p=2, dim=1)

    def temperature(self):
        return self.log_temp.exp().clamp(0.01, 1.0).item()


# ══════════════════════════════════════════════════════════════════════════════
# v12-gated: 3 независимых энкодера + softmax gate (learned per-sample)
# ══════════════════════════════════════════════════════════════════════════════
class CareerMLP_Gated(nn.Module):
    """
    Gated fusion (v12 Task B = v17 Task A):
        g = softmax( gate_net(x) )  — scalar gate per branch, per sample
        hi = encode_i(xi)            — branch-specific encoders
    out = L2_norm( sum_i g_i * proj_i(hi) )
    """
    def __init__(self, dims, hidden_dim=MLP_HIDDEN, dropout=DROPOUT_RATE,
                 emb_dim=EMB_DIM, vocab_size=1):
        super().__init__()
        assert len(dims) == 3, "Gated fusion requires exactly 3 branches"
        self.dims = dims
        total_dim = sum(dims)

        def _branch(d):
            return nn.Sequential(
                nn.Linear(d, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim//2), nn.BatchNorm1d(hidden_dim//2), nn.ReLU(),
                nn.Dropout(dropout * 0.5),
                nn.Linear(hidden_dim//2, emb_dim),
            )

        self.enc1 = _branch(dims[0])
        self.enc2 = _branch(dims[1])
        self.enc3 = _branch(dims[2])

        self.gate = nn.Sequential(
            nn.Linear(total_dim, hidden_dim//2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim//2, 3),
        )
        self.log_temp = nn.Parameter(torch.tensor(_init_log_temp(vocab_size)))

    def forward(self, x, return_gates=False):
        d0, d1, d2 = self.dims
        x1 = x[:, :d0]
        x2 = x[:, d0:d0+d1]
        x3 = x[:, d0+d1:]

        g = F.softmax(self.gate(x), dim=1)  # (B, 3)
        h1 = self.enc1(x1); h2 = self.enc2(x2); h3 = self.enc3(x3)
        out = g[:, 0:1] * h1 + g[:, 1:2] * h2 + g[:, 2:3] * h3
        out = F.normalize(out, p=2, dim=1)

        if return_gates:
            return out, g.detach().cpu()
        return out

    def temperature(self):
        return self.log_temp.exp().clamp(0.01, 1.0).item()


# ── train / predict универсальные ─────────────────────────────────────────────
def predict_mlp(model, X, re_tensor, batch_size=BATCH_SIZE):
    model.eval(); rd = re_tensor.to(device)
    ld = DataLoader(CareerDataset(X, np.zeros(len(X), dtype=np.int64)), batch_size=batch_size)
    sc = []
    with torch.no_grad():
        for Xb, _ in ld:
            t = model.log_temp.exp().clamp(0.01, 1.0)
            out = model(Xb.to(device))
            sc.append(((out @ rd.T) / t).cpu().numpy())
    return np.concatenate(sc, axis=0)

def train_mlp(model, X_tr, y_tr_, X_vl, y_vl_, re_tensor,
              epochs=MLP_EPOCHS, patience=PATIENCE, lr=3e-4, verbose=False):
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5, min_lr=1e-5)
    rd    = re_tensor.to(device)
    crit  = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    trl   = DataLoader(CareerDataset(X_tr, y_tr_), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    vll   = DataLoader(CareerDataset(X_vl, y_vl_), batch_size=BATCH_SIZE)
    best_loss = float("inf")
    best_state = {k: v.clone() for k, v in model.state_dict().items()}
    no_imp = 0

    for ep in range(epochs):
        model.train(); tl = 0.0
        for Xb, yb in trl:
            Xb, yb = Xb.to(device), yb.to(device)
            model.log_temp.data.clamp_(math.log(0.01), math.log(1.0))
            loss = crit((model(Xb) @ rd.T) / model.log_temp.exp(), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            tl += loss.item()
        model.eval(); vl = 0.0
        with torch.no_grad():
            for Xb, yb in vll:
                Xb, yb = Xb.to(device), yb.to(device)
                vl += crit((model(Xb) @ rd.T) / model.log_temp.exp(), yb).item()
        vl /= max(len(vll), 1); sched.step(vl)
        if verbose: print(f"  Ep{ep+1:2d}: tr={tl/len(trl):.4f} val={vl:.4f} T={model.temperature():.4f}")
        if vl < best_loss:
            best_loss = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= patience: break
    model.load_state_dict(best_state)
    return model

def norm_scores(sc):
    mn = sc.min(axis=1, keepdims=True); mx = sc.max(axis=1, keepdims=True)
    return (sc - mn) / (mx - mn + 1e-9)

print(f"CareerMLP_Additive и CareerMLP_Gated определены.")
print(f"Temperature clamp: [0.01, 1.0], init=log(1/sqrt(VOCAB_B))")
print(f"label_smoothing={LABEL_SMOOTHING}, N_FOLDS={N_FOLDS}")


CareerMLP_Additive и CareerMLP_Gated определены.
Temperature clamp: [0.01, 1.0], init=log(1/sqrt(VOCAB_B))
label_smoothing=0.1, N_FOLDS=5


## Шаг 10. Feature Ablation Study

Аблейшн в **четырёх блоках** с явными reference lines.

**Протокол v14 (retrain ablation)**:  
- Каждая конфигурация обучается заново на `idx_tr_B`, оценивается на `idx_vl_B` и `idx_te_B`.
- Seeds: `ABL_SEEDS = [42, 123, 777]` (идентичны финальному обучению).
- σ по 3 seeds измеряет только training stochasticity; для оценки статистической значимости  
  разностей между конфигурациями используется paired bootstrap в Шаге 13.

| Блок | Описание | База (Δ от) |
|---|---|---|
| A | Текстовые признаки | A0: titles only |
| B | Доп. эмбеддинги | A0: titles only |
| C | Структурные по отдельности | A1: +desc |
| D | Полные конфигурации | B5: +desc+skills+about |


In [12]:
def make_X(feat_key, struct_keys=None):
    parts = [feats[feat_key]]
    struct_map = {
        "X_skills_emb": feats["X_skills_emb"],
        "X_about_emb":  feats["X_about_emb"],
        "X_career":     X_career_B_sc,
        "X_location":   X_location_B_sc,
        "X_edu":        X_edu_B_sc,
    }
    if struct_keys:
        for k in struct_keys: parts.append(struct_map[k])
    return np.concatenate(parts, axis=1).astype(np.float32)

ABLATION_CONFIGS = [
    # Блок A
    ("A0: base (titles only)",     "X_seq", None),
    ("A1: +desc",                  "X_doc", None),
    # Блок B
    ("B1: +skills_emb",            "X_seq", ["X_skills_emb"]),
    ("B2: +about_emb",             "X_seq", ["X_about_emb"]),
    ("B3: +desc+skills",           "X_doc", ["X_skills_emb"]),
    ("B4: +desc+about",            "X_doc", ["X_about_emb"]),
    ("B5: +desc+skills+about",     "X_doc", ["X_skills_emb", "X_about_emb"]),
    # Блок C
    ("C1: +desc+career",           "X_doc", ["X_career"]),
    ("C2: +desc+location",         "X_doc", ["X_location"]),
    ("C3: +desc+edu",              "X_doc", ["X_edu"]),
    ("C4: +desc+career+location",  "X_doc", ["X_career", "X_location"]),
    ("C5: +desc+career+edu",       "X_doc", ["X_career", "X_edu"]),
    ("C6: +desc+location+edu",     "X_doc", ["X_location", "X_edu"]),
    ("C7: +desc+struct (all)",     "X_doc", ["X_career", "X_location", "X_edu"]),
    # Блок D
    ("D1: +best+struct",           "X_doc", ["X_skills_emb", "X_career", "X_location", "X_edu"]),
    ("D2: +best+about+struct",     "X_doc", ["X_skills_emb", "X_about_emb", "X_career", "X_location", "X_edu"]),
]

# v14: используем ABL_SEEDS (идентичны MLP_KFOLD_SEEDS)
print(f"Ablation: {len(ABLATION_CONFIGS)} конфигурации × {N_SEEDS_ABL} seeds {ABL_SEEDS}")
print("Архитектура: CareerMLP_Additive (retrain ablation)")
print("Протокол: каждая конфигурация обучается заново, веса не фиксируются между конфигурациями")
ablation_results = {}

for cfg_name, feat_key, struct_keys in ABLATION_CONFIGS:
    X_cfg = make_X(feat_key, struct_keys)
    dim   = X_cfg.shape[1]
    mrrs, r1s, r3s, r5s, val_mrrs = [], [], [], [], []
    for seed in ABL_SEEDS:  # v14: ABL_SEEDS вместо range(N_SEEDS_ABL)
        torch.manual_seed(seed)
        model = CareerMLP_Additive(dim, emb_dim=EMB_DIM, vocab_size=VOCAB_B).to(device)
        model = train_mlp(model, X_cfg[idx_tr_B], y_tr_B, X_cfg[idx_vl_B], y_vl_B, role_emb)
        sc_te = predict_mlp(model, X_cfg[idx_te_B], role_emb)
        sc_vl = predict_mlp(model, X_cfg[idx_vl_B], role_emb)
        res_te = eval_full(sc_te, y_te_B)
        res_vl = eval_full(sc_vl, y_vl_B)
        mrrs.append(res_te["MRR"]); r1s.append(res_te["R@1"])
        r3s.append(res_te["R@3"]); r5s.append(res_te["R@5"])
        val_mrrs.append(res_vl["MRR"])
        print(f"  {cfg_name} [dim={dim}] seed={seed}: {fmt(res_te)} | val_MRR={res_vl['MRR']*100:.2f}%")
        del model; _gc()
    ablation_results[cfg_name] = {
        "MRR": np.mean(mrrs), "MRR_std": np.std(mrrs),
        "val_MRR": np.mean(val_mrrs), "val_MRR_std": np.std(val_mrrs),
        "R@1": np.mean(r1s),  "R@3": np.mean(r3s), "R@5": np.mean(r5s),
        "block": cfg_name[0],
    }

# Добавляем Linear baselines в таблицу
for cfg_name, v in results_linear.items():
    ablation_results[cfg_name] = {
        "MRR": v["test"]["MRR"], "MRR_std": 0.0,
        "R@1": v["test"]["R@1"], "R@3": v["test"]["R@3"], "R@5": v["test"]["R@5"],
        "block": "Linear",
    }

BLOCK_LABELS = {
    "Linear": "Linear baselines",
    "A": "Блок A — текстовые признаки  (Δ vs A0: base)",
    "B": "Блок B — доп. эмбеддинги     (Δ vs A0: base)",
    "C": "Блок C — структурные признаки по отдельности  (Δ vs A1: +desc)",
    "D": "Блок D — полные конфигурации  (Δ vs B5: +desc+skills+about)",
}

def _base_mrr_for_block(block):
    if block == "C": return ablation_results.get("A1: +desc", {}).get("MRR", 0)
    if block == "D": return ablation_results.get("B5: +desc+skills+about", {}).get("MRR", 0)
    return ablation_results.get("A0: base (titles only)", {}).get("MRR", 0)

print("\n" + "=" * 88)
print("ABLATION SUMMARY (Task B — job_1_position_norm) [v14-additive, retrain]:")
cur_block = None
all_cfg_order = list(results_linear.keys()) + [c for c, _, _ in ABLATION_CONFIGS]
for cfg_name in all_cfg_order:
    v = ablation_results[cfg_name]; block = v.get("block", "Linear")
    if block != cur_block:
        cur_block = block
        print(f"\n  ┌── {BLOCK_LABELS.get(block, block)} ──")
        if block in ("C","D"):
            base_ref = _base_mrr_for_block(block)
            label = "A1: +desc" if block=="C" else "B5: +desc+skills+about"
            print(f"  │   база: {label} = {base_ref*100:.2f}%")
        print(f"  │   {'Конфигурация':<35s} {'MRR':>8s}  {'±std':>7s}  {'R@1':>7s}  {'R@3':>7s}  {'ΔMRR':>8s}")
        print(f"  │   " + "-"*68)
    std_str = f"±{v['MRR_std']*100:.2f}" if v["MRR_std"] > 0 else "(det.) "
    base_mrr = _base_mrr_for_block(block); delta = v["MRR"] - base_mrr
    REF_CFGS = ("A0: base (titles only)", "A1: +desc", "B5: +desc+skills+about")
    d_str = "  [REF]  " if cfg_name in REF_CFGS else f"({delta*100:+.2f})"
    print(f"  │   {cfg_name:<35s} {v['MRR']*100:7.2f}%  {std_str:<7s}  {v['R@1']*100:6.2f}%  {d_str}")

df_abl = pd.DataFrame({
    k: {"block": v.get("block","Linear"), "MRR(%)": round(v["MRR"]*100,2),
        "std": round(v["MRR_std"]*100,2), "R@1(%)": round(v["R@1"]*100,2),
        "R@3(%)": round(v["R@3"]*100,2), "R@5(%)": round(v["R@5"]*100,2)}
    for k, v in ablation_results.items()
}).T
df_abl.to_csv("ablation_taskB_v14.csv")
print("\nСохранено: ablation_taskB_v14.csv")


Ablation: 16 конфигурации × 3 seeds [42, 123, 777]
Архитектура: CareerMLP_Additive (retrain ablation)
Протокол: каждая конфигурация обучается заново, веса не фиксируются между конфигурациями
  A0: base (titles only) [dim=1024] seed=42: MRR=63.37%  R@1=48.39%  R@3=73.69%  R@5=83.07% | val_MRR=63.54%
  A0: base (titles only) [dim=1024] seed=123: MRR=63.40%  R@1=48.50%  R@3=73.83%  R@5=83.05% | val_MRR=63.55%
  A0: base (titles only) [dim=1024] seed=777: MRR=63.46%  R@1=48.57%  R@3=73.71%  R@5=83.11% | val_MRR=63.47%
  A1: +desc [dim=1024] seed=42: MRR=65.72%  R@1=50.16%  R@3=77.22%  R@5=86.21% | val_MRR=65.67%
  A1: +desc [dim=1024] seed=123: MRR=65.51%  R@1=49.76%  R@3=77.30%  R@5=86.25% | val_MRR=65.67%
  A1: +desc [dim=1024] seed=777: MRR=65.70%  R@1=50.05%  R@3=77.55%  R@5=86.48% | val_MRR=65.54%
  B1: +skills_emb [dim=2048] seed=42: MRR=66.99%  R@1=51.76%  R@3=78.45%  R@5=87.27% | val_MRR=67.05%
  B1: +skills_emb [dim=2048] seed=123: MRR=67.11%  R@1=51.87%  R@3=78.82%  R@5=87.31% | 

## Шаг 11. Выбор лучшей конфигурации

In [13]:
# Выбор лучшей конфигурации по val_MRR (не по test, чтобы избежать утечки)
best_cfg_name = max(
    [(c, ablation_results[c]["val_MRR"]) for c, _, _ in ABLATION_CONFIGS],
    key=lambda x: x[1]
)[0]
best_tuple = next((c, fk, sk) for c, fk, sk in ABLATION_CONFIGS if c == best_cfg_name)
_, BEST_FEAT_KEY, BEST_STRUCT_KEYS = best_tuple

# Fallback к A1: +desc если она не хуже лучшей более чем на 0.3pp (по val)
ref_desc_val = ablation_results.get("A1: +desc", {}).get("val_MRR", 0)
if ref_desc_val >= ablation_results[best_cfg_name]["val_MRR"] - 0.003:
    best_cfg_name = "A1: +desc"
    BEST_FEAT_KEY = "X_doc"
    BEST_STRUCT_KEYS = None

X_best = make_X(BEST_FEAT_KEY, BEST_STRUCT_KEYS)
print(f"Лучшая конфигурация (выбрана по val): '{best_cfg_name}', dim={X_best.shape[1]}")
print(f"  val_MRR={ablation_results[best_cfg_name]['val_MRR']*100:.2f}%  "
      f"(test_MRR={ablation_results[best_cfg_name]['MRR']*100:.2f}%  "
      f"R@1={ablation_results[best_cfg_name]['R@1']*100:.2f}%)")

print("\nТоп-5 конфигураций по val_MRR:")
top5 = sorted(
    [(c, ablation_results[c]["val_MRR"]) for c, _, _ in ABLATION_CONFIGS],
    key=lambda x: -x[1]
)[:5]
for rank, (name, val_mrr) in enumerate(top5, 1):
    test_mrr = ablation_results[name]["MRR"]
    print(f"  {rank}. {name:<38s} val_MRR={val_mrr*100:.2f}%  test_MRR={test_mrr*100:.2f}%")


Лучшая конфигурация (выбрана по val): 'D2: +best+about+struct', dim=3081
  val_MRR=68.50%  (test_MRR=68.21%  R@1=52.74%)

Топ-5 конфигураций по val_MRR:
  1. D2: +best+about+struct                 val_MRR=68.50%  test_MRR=68.21%
  2. B5: +desc+skills+about                 val_MRR=68.23%  test_MRR=68.21%
  3. D1: +best+struct                       val_MRR=67.73%  test_MRR=67.55%
  4. B3: +desc+skills                       val_MRR=67.49%  test_MRR=67.43%
  5. B1: +skills_emb                        val_MRR=67.13%  test_MRR=67.00%


## Шаг 12. MLP K-Fold × Multi-Seed (v12-additive)


In [14]:
print(f"MLP K-Fold × {len(MLP_KFOLD_SEEDS)} seeds (K={N_FOLDS}, '{best_cfg_name}', dim={X_best.shape[1]}):")
print(f"Архитектура: CareerMLP_Additive")
print(f"Всего моделей: {N_FOLDS * len(MLP_KFOLD_SEEDS)}")

all_fold_scores    = []
all_fold_scores_vl = []

for seed in MLP_KFOLD_SEEDS:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    seed_scores_te = []
    seed_scores_vl = []
    for fold, (ti, vi) in enumerate(skf.split(X_best[idx_tv_B], y_tv_B)):
        torch.manual_seed(seed * 100 + fold)
        m = CareerMLP_Additive(X_best.shape[1], emb_dim=EMB_DIM, vocab_size=VOCAB_B).to(device)
        m = train_mlp(m, X_best[idx_tv_B][ti], y_tv_B[ti],
                         X_best[idx_tv_B][vi], y_tv_B[vi], role_emb)
        sc_te = predict_mlp(m, X_best[idx_te_B], role_emb)
        sc_vl = predict_mlp(m, X_best[idx_vl_B], role_emb)
        seed_scores_te.append(sc_te)
        seed_scores_vl.append(sc_vl)
        res = eval_full(sc_te, y_te_B)
        print(f"  seed={seed} fold={fold+1:2d}: {fmt(res)}")
        del m; _gc()
    all_fold_scores.extend(seed_scores_te)
    all_fold_scores_vl.extend(seed_scores_vl)
    sc_seed_avg = np.mean(seed_scores_te, axis=0)
    print(f"  >>> seed={seed} avg ({N_FOLDS} folds): {fmt(eval_full(sc_seed_avg, y_te_B))}")

sc_mlp_kfold    = np.mean(all_fold_scores, axis=0)
sc_mlp_kfold_vl = np.mean(all_fold_scores_vl, axis=0)
res_mlp_kfold   = eval_full(sc_mlp_kfold, y_te_B)
print(f"\n>>> MLP K-Fold × {len(MLP_KFOLD_SEEDS)} seeds ({N_FOLDS} folds each): {fmt(res_mlp_kfold)}")


MLP K-Fold × 3 seeds (K=5, 'D2: +best+about+struct', dim=3081):
Архитектура: CareerMLP_Additive
Всего моделей: 15
  seed=42 fold= 1: MRR=68.32%  R@1=53.26%  R@3=80.21%  R@5=88.14%
  seed=42 fold= 2: MRR=68.29%  R@1=52.97%  R@3=80.40%  R@5=88.33%
  seed=42 fold= 3: MRR=68.33%  R@1=53.11%  R@3=80.08%  R@5=88.34%
  seed=42 fold= 4: MRR=68.17%  R@1=52.85%  R@3=80.28%  R@5=88.40%
  seed=42 fold= 5: MRR=68.00%  R@1=52.46%  R@3=80.30%  R@5=88.33%
  >>> seed=42 avg (5 folds): MRR=68.70%  R@1=53.29%  R@3=80.82%  R@5=88.86%
  seed=123 fold= 1: MRR=68.48%  R@1=53.30%  R@3=80.43%  R@5=88.58%
  seed=123 fold= 2: MRR=67.92%  R@1=52.48%  R@3=80.07%  R@5=88.40%
  seed=123 fold= 3: MRR=68.14%  R@1=52.70%  R@3=80.34%  R@5=88.50%
  seed=123 fold= 4: MRR=68.09%  R@1=52.74%  R@3=80.05%  R@5=88.11%
  seed=123 fold= 5: MRR=68.23%  R@1=53.08%  R@3=80.05%  R@5=88.29%
  >>> seed=123 avg (5 folds): MRR=68.70%  R@1=53.28%  R@3=80.75%  R@5=88.90%
  seed=777 fold= 1: MRR=68.19%  R@1=52.82%  R@3=80.23%  R@5=88.42%
 

## Шаг 12b. MLP K-Fold × Multi-Seed (v14-gated) — Routing Analysis

Gated-версия с 3 ветвями:
- Branch 1: career history embeddings (X_doc)
- Branch 2: skills embeddings (X_skills_emb)
- Branch 3: structural features (X_career + X_location + X_edu)

**Методологический дисклеймер (замечание 2.6/28)**:  
Среднее gate-значение `ḡ_k = E[g_k(x)]` отражает среднюю долю ветви в финальном векторе,  
но **не является** feature importance. Причины:
1. `g_k` зависит от масштаба output конкретного branch encoder, а не только от информативности признаков.
2. Ветвь с малым `ḡ_k` может быть критичной в редких случаях.
3. Gate-веса не контролируют корреляцию между ветвями.

Gate-анализ используется как **диагностика routing** — на какую ветвь модель "полагается" в среднем.  
Для оценки feature importance используется retrain ablation (Шаг 10).


In [15]:
EMB_DIM_BRANCH = EMB_DIM  # 1024 (career emb)
SKILLS_DIM     = feats["X_skills_emb"].shape[1]   # 1024
STRUCT_DIM_G   = X_career_B_sc.shape[1] + X_location_B_sc.shape[1] + X_edu_B_sc.shape[1]

X_gated = np.concatenate([
    feats["X_doc"],
    feats["X_skills_emb"],
    np.concatenate([X_career_B_sc, X_location_B_sc, X_edu_B_sc], axis=1),
], axis=1).astype(np.float32)

GATED_DIMS = [EMB_DIM_BRANCH, SKILLS_DIM, STRUCT_DIM_G]
print(f"Gated dims: {GATED_DIMS}, total={sum(GATED_DIMS)}")
print(f"MLP Gated × {len(MLP_KFOLD_SEEDS)} seeds (K={N_FOLDS}):")

all_fold_scores_gated    = []
all_fold_scores_gated_vl = []
all_gate_weights         = []

for seed in MLP_KFOLD_SEEDS:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    seed_sc_te, seed_sc_vl = [], []
    for fold, (ti, vi) in enumerate(skf.split(X_gated[idx_tv_B], y_tv_B)):
        torch.manual_seed(seed * 100 + fold)
        m_g = CareerMLP_Gated(GATED_DIMS, emb_dim=EMB_DIM, vocab_size=VOCAB_B).to(device)
        m_g = train_mlp(m_g, X_gated[idx_tv_B][ti], y_tv_B[ti],
                             X_gated[idx_tv_B][vi], y_tv_B[vi], role_emb)
        sc_te = predict_mlp(m_g, X_gated[idx_te_B], role_emb)
        sc_vl = predict_mlp(m_g, X_gated[idx_vl_B], role_emb)
        seed_sc_te.append(sc_te); seed_sc_vl.append(sc_vl)
        res = eval_full(sc_te, y_te_B)
        print(f"  seed={seed} fold={fold+1:2d}: {fmt(res)}")

        m_g.eval()
        with torch.no_grad():
            ld_g = DataLoader(CareerDataset(X_gated[idx_te_B], np.zeros(len(idx_te_B))),
                              batch_size=BATCH_SIZE)
            gate_batches = []
            for Xb, _ in ld_g:
                _, g = m_g(Xb.to(device), return_gates=True)
                gate_batches.append(g)
        all_gate_weights.append(torch.cat(gate_batches, dim=0).numpy())
        del m_g; _gc()

    all_fold_scores_gated.extend(seed_sc_te)
    all_fold_scores_gated_vl.extend(seed_sc_vl)
    sc_seed_avg = np.mean(seed_sc_te, axis=0)
    print(f"  >>> seed={seed} avg: {fmt(eval_full(sc_seed_avg, y_te_B))}")

sc_mlp_gated    = np.mean(all_fold_scores_gated, axis=0)
sc_mlp_gated_vl = np.mean(all_fold_scores_gated_vl, axis=0)
res_mlp_gated   = eval_full(sc_mlp_gated, y_te_B)
print(f"\n>>> MLP Gated × {len(MLP_KFOLD_SEEDS)} seeds ({N_FOLDS} folds each): {fmt(res_mlp_gated)}")

gate_mean = np.mean([g for g in all_gate_weights], axis=0)
gate_avg  = gate_mean.mean(axis=0)
gate_std  = gate_mean.std(axis=0)
branch_names = ["career_emb", "skills_emb", "struct"]
print("\nGate analysis (mean ± std across test set):")
for i, (name, avg, std) in enumerate(zip(branch_names, gate_avg, gate_std)):
    print(f"  g_{i+1} ({name:<15s}): {avg:.3f} ± {std:.3f}")
print("Интерпретация: g_i > 1/3 → ветвь получила больше веса, чем при uniform fusion")


Gated dims: [1024, 1024, 9], total=2057
MLP Gated × 3 seeds (K=5):
  seed=42 fold= 1: MRR=67.83%  R@1=52.52%  R@3=79.42%  R@5=88.29%
  seed=42 fold= 2: MRR=67.77%  R@1=52.39%  R@3=79.93%  R@5=88.11%
  seed=42 fold= 3: MRR=67.70%  R@1=52.31%  R@3=79.78%  R@5=88.38%
  seed=42 fold= 4: MRR=67.65%  R@1=52.38%  R@3=79.46%  R@5=88.13%
  seed=42 fold= 5: MRR=67.48%  R@1=51.99%  R@3=79.68%  R@5=88.13%
  >>> seed=42 avg: MRR=68.07%  R@1=52.65%  R@3=80.12%  R@5=88.75%
  seed=123 fold= 1: MRR=67.51%  R@1=51.95%  R@3=79.70%  R@5=88.08%
  seed=123 fold= 2: MRR=67.56%  R@1=52.13%  R@3=79.68%  R@5=87.94%
  seed=123 fold= 3: MRR=67.81%  R@1=52.38%  R@3=80.02%  R@5=88.37%
  seed=123 fold= 4: MRR=67.66%  R@1=52.27%  R@3=79.88%  R@5=88.12%
  seed=123 fold= 5: MRR=67.60%  R@1=52.17%  R@3=79.64%  R@5=88.34%
  >>> seed=123 avg: MRR=68.00%  R@1=52.46%  R@3=80.30%  R@5=88.68%
  seed=777 fold= 1: MRR=67.67%  R@1=52.21%  R@3=79.69%  R@5=88.31%
  seed=777 fold= 2: MRR=67.79%  R@1=52.46%  R@3=79.71%  R@5=88.17%
 

## Шаг 13. Entropy-Stratified Evaluation (v14 — bootstrap CI)

**Научная гипотеза H2:**  
`Δ MRR(+desc − base)` растёт от Low → High entropy.  
Описания должностей наиболее полезны при **неоднозначных переходах**.

**Статистические тесты v14 (замечания 38/39/46)**:
- p-value оценивается как доля bootstrap-разностей ≤ 0 (не t-test на bootstrap-репликах)
- Тест монотонности: `(ΔMid − ΔLow) > 0` и `(ΔHigh − ΔMid) > 0` с bootstrap CI
- Контроль за частотностью: параллельная стратификация по support (анализ F в ячейке 35)

**Ограничение (замечание 1.5/37)**:  
H(l) оценивается по обучающей выборке. Редкие роли → мало наблюдений → зашумлённая оценка H →  
потенциальная конфаундированность с частотностью. Анализ F проверяет, воспроизводится ли  
паттерн при стратификации по support вместо entropy.


In [16]:
# ── Энтропия переходов (обучающее множество) ──────────────────────────────────
trans_cnt = np.zeros((VOCAB_B, VOCAB_B), dtype=np.float32)
for i in idx_tr_B:
    j2  = str(df_B.iloc[i].get(JOB2_COL_B, "") or "").strip()
    tgt = y_B[i]
    if j2 in role_to_id_B and tgt in role_to_id_B:
        trans_cnt[role_to_id_B[j2], role_to_id_B[tgt]] += 1

trans_prob   = (trans_cnt + 1.0) / (trans_cnt + 1.0).sum(axis=1, keepdims=True)
role_entropy = np.array([scipy_entropy(trans_prob[i]) for i in range(VOCAB_B)])

q33, q66 = np.percentile(role_entropy, [33, 66])
mean_H   = float(np.mean(role_entropy))
print(f"Энтропия переходов: Q33={q33:.3f}  Q66={q66:.3f}  mean={mean_H:.3f}")

sorted_ent = sorted(enumerate(all_roles_B), key=lambda x: role_entropy[x[0]])
print("Low H (предсказуемые должности):")
[print(f"  {r:45s}: H={role_entropy[rid]:.3f}") for rid,r in sorted_ent[:5]]
print("High H (неоднозначные должности):")
[print(f"  {r:45s}: H={role_entropy[rid]:.3f}") for rid,r in sorted_ent[-5:]]

low_idx, mid_idx, high_idx = [], [], []
for j, i in enumerate(idx_te_B):
    j2 = str(df_B.iloc[i].get(JOB2_COL_B, "") or "").strip()
    H  = role_entropy[role_to_id_B[j2]] if j2 in role_to_id_B else mean_H
    if H <= q33: low_idx.append(j)
    elif H <= q66: mid_idx.append(j)
    else: high_idx.append(j)

low_idx  = np.array(low_idx,  dtype=np.int64)
mid_idx  = np.array(mid_idx,  dtype=np.int64)
high_idx = np.array(high_idx, dtype=np.int64)

MIN_GROUP_SIZE = 30
print(f"\nTest группы: Low={len(low_idx)}, Mid={len(mid_idx)}, High={len(high_idx)}")
group_valid = {}
for gname, gidx in [("Low", low_idx), ("Mid", mid_idx), ("High", high_idx)]:
    valid = len(gidx) >= MIN_GROUP_SIZE
    group_valid[gname] = valid
    status = "✓" if valid else "✗ СЛИШКОМ МАЛА"
    print(f"  {gname}: n={len(gidx)} {status}")

expected = len(y_te_B) / 3
for gname, gidx in [("Low", low_idx), ("Mid", mid_idx), ("High", high_idx)]:
    ratio = len(gidx) / expected
    if ratio < 0.7 or ratio > 1.3:
        print(f"  ⚠ {gname}: размер {ratio:.2f}x от равномерного")


def bootstrap_mrr(scores, y_true, gidx, n_boot=N_BOOTSTRAP, seed=BOOTSTRAP_SEED):
    if len(gidx) < MIN_GROUP_SIZE:
        return np.nan, (np.nan, np.nan), []
    rng = np.random.default_rng(seed)
    sc_g = scores[gidx]; y_g = y_true[gidx]
    point = eval_full(sc_g, y_g)["MRR"]
    samples = []
    for _ in range(n_boot):
        ridx = rng.integers(0, len(gidx), size=len(gidx))
        samples.append(eval_full(sc_g[ridx], y_g[ridx])["MRR"])
    alpha = 1 - CI_LEVEL
    ci = (np.percentile(samples, 100*alpha/2), np.percentile(samples, 100*(1-alpha/2)))
    return point, ci, samples

print(f"\nBootstrap CI ({N_BOOTSTRAP} iterations, {CI_LEVEL*100:.0f}% CI)...")


Энтропия переходов: Q33=2.096  Q66=2.559  mean=2.289
Low H (предсказуемые должности):
  Product Designer (UI/UX)                     : H=0.810
  Frontend Developer                           : H=1.580
  Mobile Developer                             : H=1.631
  1C Developer / Architect                     : H=1.741
  System Administrator                         : H=1.812
High H (неоднозначные должности):
  Security Architect                           : H=2.792
  Data Engineer                                : H=2.848
  Business Intelligence (BI) Developer         : H=3.007
  Platform Engineer                            : H=3.038
  Digital Transformation Manager               : H=3.072

Test группы: Low=4943, Mid=3458, High=436
  Low: n=4943 ✓
  Mid: n=3458 ✓
  High: n=436 ✓
  ⚠ Low: размер 1.68x от равномерного
  ⚠ High: размер 0.15x от равномерного

Bootstrap CI (1000 iterations, 95% CI)...


In [17]:
# ── Multi-seed: обучаем base и +desc (ABL_SEEDS) ────────────────────────────
sc_base_seeds = []; sc_desc_seeds = []

for seed in ABL_SEEDS:  # v14: ABL_SEEDS
    torch.manual_seed(seed)
    m_b = CareerMLP_Additive(feats["X_seq"].shape[1], emb_dim=EMB_DIM, vocab_size=VOCAB_B).to(device)
    m_b = train_mlp(m_b, feats["X_seq"][idx_tr_B], y_tr_B,
                         feats["X_seq"][idx_vl_B], y_vl_B, role_emb)
    sc_base_seeds.append(predict_mlp(m_b, feats["X_seq"][idx_te_B], role_emb))
    del m_b; _gc()

    torch.manual_seed(seed)
    m_d = CareerMLP_Additive(feats["X_doc"].shape[1], emb_dim=EMB_DIM, vocab_size=VOCAB_B).to(device)
    m_d = train_mlp(m_d, feats["X_doc"][idx_tr_B], y_tr_B,
                         feats["X_doc"][idx_vl_B], y_vl_B, role_emb)
    sc_desc_seeds.append(predict_mlp(m_d, feats["X_doc"][idx_te_B], role_emb))
    del m_d; _gc()

sc_base_te = np.mean(sc_base_seeds, axis=0)
sc_desc_te = np.mean(sc_desc_seeds, axis=0)

# ── Entropy-stratified таблица с Bootstrap CI ─────────────────────────────────
ent_scores_dict = {
    "base (MLP titles, 3s)":         sc_base_te,
    "+desc (MLP, 3s)":               sc_desc_te,
    "MLP KFold (additive)":          sc_mlp_kfold,
    "MLP KFold (gated)":             sc_mlp_gated,
}

groups_dict = {
    "Low":  low_idx,
    "Mid":  mid_idx,
    "High": high_idx,
    "All":  np.arange(len(y_te_B))
}

print("\nEntropy-stratified MRR (%) с Bootstrap 95% CI:")
print(f"  {'Модель':<40s}  {'Low MRR (95%CI)':>22s}  {'Mid MRR (95%CI)':>22s}  {'High MRR (95%CI)':>22s}  {'All':>7s}")
print("  " + "-"*110)

entropy_rows = {}
entropy_boot = {}

for name, sc in ent_scores_dict.items():
    row_mrr = {}
    row_ci  = {}
    for gname, gidx in groups_dict.items():
        pt, ci, _ = bootstrap_mrr(sc, y_te_B, gidx)
        row_mrr[gname] = round(pt*100, 2) if not np.isnan(pt) else float("nan")
        row_ci[gname]  = (round(ci[0]*100, 2), round(ci[1]*100, 2))
    entropy_rows[name] = row_mrr
    entropy_boot[name] = row_ci

    def _fmt_ci(gname):
        pt = row_mrr[gname]; ci = row_ci[gname]
        if np.isnan(pt): return "   N/A (n<30)  "
        return f"{pt:.1f}% [{ci[0]:.1f},{ci[1]:.1f}]"

    print(f"  {name:<40s}  {_fmt_ci('Low'):>22s}  {_fmt_ci('Mid'):>22s}  {_fmt_ci('High'):>22s}  {row_mrr['All']:>6.1f}%")

# ── Δ MRR(+desc − base) — bootstrap p-value (v14: исправлен тест) ────────────
print("\n--- Δ MRR(+desc − base) — Гипотеза H2: Δ растёт Low→High ---")
print(f"  Bootstrap p-value: p = доля bootstrap-разностей ≤ 0 (односторонний)")
print(f"  (N_BOOTSTRAP={N_BOOTSTRAP}, seed={BOOTSTRAP_SEED+1})")
print(f"  {'Группа':5s}  {'n':>5s}  {'ΔMRR':>8s}  {'95%CI':>18s}  {'p-value':>10s}  {'Значим?':>10s}")
print("  " + "-"*70)

delta_by_group = {}
delta_boot_samples = {}  # сохраняем для теста монотонности

for gname, gidx in [("Low", low_idx), ("Mid", mid_idx), ("High", high_idx)]:
    if len(gidx) < MIN_GROUP_SIZE:
        print(f"  {gname:5s}  {len(gidx):5d}  — слишком мала группа —")
        delta_by_group[gname] = float("nan")
        delta_boot_samples[gname] = []
        continue

    rng = np.random.default_rng(BOOTSTRAP_SEED + 1)
    sc_base_g = sc_base_te[gidx]; sc_desc_g = sc_desc_te[gidx]; y_g = y_te_B[gidx]

    deltas = []
    for _ in range(N_BOOTSTRAP):
        ridx = rng.integers(0, len(gidx), size=len(gidx))
        d = (eval_full(sc_desc_g[ridx], y_g[ridx])["MRR"]
             - eval_full(sc_base_g[ridx], y_g[ridx])["MRR"])
        deltas.append(d)
    deltas = np.array(deltas)
    delta_boot_samples[gname] = deltas

    delta_point = (entropy_rows["+desc (MLP, 3s)"][gname]
                   - entropy_rows["base (MLP titles, 3s)"][gname])
    alpha = 1 - CI_LEVEL
    ci = (np.percentile(deltas, 100*alpha/2), np.percentile(deltas, 100*(1-alpha/2)))

    # v14: bootstrap p-value вместо t-test (замечание 39)
    # p = доля bootstrap-разностей ≤ 0 (H0: Δ ≤ 0, H1: Δ > 0 — односторонний)
    p_val = float(np.mean(deltas <= 0))
    signif = "✓ значим" if p_val < (1 - CI_LEVEL) else "— н.з."
    delta_by_group[gname] = delta_point
    print(f"  {gname:5s}  {len(gidx):5d}  {delta_point:+7.2f}pp  "
          f"[{ci[0]*100:+.2f},{ci[1]*100:+.2f}]  p={p_val:.4f}   {signif}")

# ── Тест монотонности: (ΔMid − ΔLow) > 0 и (ΔHigh − ΔMid) > 0 ─────────────
print("\n--- Тест монотонности (v14 — замечание 38/46) ---")
print(f"  H0_1: ΔMid = ΔLow    H1_1: ΔMid > ΔLow")
print(f"  H0_2: ΔHigh = ΔMid   H1_2: ΔHigh > ΔMid")

for (g1, g2), label in [
    (("Low", "Mid"),  "ΔMid − ΔLow"),
    (("Mid", "High"), "ΔHigh − ΔMid"),
]:
    s1 = delta_boot_samples.get(g1, [])
    s2 = delta_boot_samples.get(g2, [])
    if len(s1) == 0 or len(s2) == 0:
        print(f"  {label}: — недостаточно данных —")
        continue
    # Попарный bootstrap: разность разностей
    min_len = min(len(s1), len(s2))
    diff_of_diffs = s2[:min_len] - s1[:min_len]
    point_val = delta_by_group.get(g2, np.nan) - delta_by_group.get(g1, np.nan)
    alpha = 1 - CI_LEVEL
    ci_dd = (np.percentile(diff_of_diffs, 100*alpha/2),
             np.percentile(diff_of_diffs, 100*(1-alpha/2)))
    # p-value: доля diff_of_diffs <= 0
    p_mono = float(np.mean(diff_of_diffs <= 0))
    signif = "✓ значим" if p_mono < (1 - CI_LEVEL) else "— н.з."
    print(f"  {label:<18s}: point={point_val*100:+.2f}pp  "
          f"95%CI=[{ci_dd[0]*100:+.2f},{ci_dd[1]*100:+.2f}]  p={p_mono:.4f}  {signif}")

# ── Итоговое заключение по H2 ─────────────────────────────────────────────────
low_d  = delta_by_group.get("Low",  float("nan"))
mid_d  = delta_by_group.get("Mid",  float("nan"))
high_d = delta_by_group.get("High", float("nan"))
if not any(np.isnan(x) for x in [low_d, mid_d, high_d]):
    monotone = (mid_d >= low_d) and (high_d >= mid_d)
    if monotone:
        print(f"\n✓ Паттерн монотонен: Δ(Low)={low_d:+.2f}pp ≤ Δ(Mid)={mid_d:+.2f}pp ≤ Δ(High)={high_d:+.2f}pp")
        print(f"  Статистическая значимость — см. тест монотонности выше.")
    else:
        print(f"\n✗ Паттерн НЕ монотонен: Δ(Low)={low_d:+.2f}pp  Δ(Mid)={mid_d:+.2f}pp  Δ(High)={high_d:+.2f}pp")

df_ent = pd.DataFrame(entropy_rows).T
df_ent_ci = pd.DataFrame({
    name: {f"{g}_ci": str(entropy_boot[name][g]) for g in groups_dict}
    for name in ent_scores_dict
}).T
df_ent.to_csv("entropy_taskB_v14.csv")
df_ent_ci.to_csv("entropy_taskB_v14_ci.csv")
print("\nСохранено: entropy_taskB_v14.csv, entropy_taskB_v14_ci.csv")



Entropy-stratified MRR (%) с Bootstrap 95% CI:
  Модель                                           Low MRR (95%CI)         Mid MRR (95%CI)        High MRR (95%CI)      All
  --------------------------------------------------------------------------------------------------------------
  base (MLP titles, 3s)                          69.4% [68.4,70.4]       56.8% [55.6,58.0]       48.4% [45.1,52.0]    63.4%
  +desc (MLP, 3s)                                70.9% [69.9,71.8]       60.1% [58.9,61.4]       54.2% [50.9,57.8]    65.8%
  MLP KFold (additive)                           73.0% [72.0,73.9]       64.1% [62.8,65.3]       57.2% [53.6,60.6]    68.7%
  MLP KFold (gated)                              72.5% [71.7,73.4]       63.1% [61.9,64.2]       58.4% [55.1,61.9]    68.1%

--- Δ MRR(+desc − base) — Гипотеза H2: Δ растёт Low→High ---
  Bootstrap p-value: p = доля bootstrap-разностей ≤ 0 (односторонний)
  (N_BOOTSTRAP=1000, seed=1)
  Группа      n      ΔMRR               95%CI     p-value 

## Шаг 14. Итоги


In [18]:
print("=" * 80)
print("ИТОГИ — Career Path Prediction Task B v14 (IT domain, RU)")
print("=" * 80)
print(f"Энкодер:   {MODEL_NAME} (LAST стратегия, R1={FT_EPOCHS_R1} эпохи + R2 hard negatives)")
print(f"           R1+R2 обучены ТОЛЬКО на train (без val) — v14")
print(f"Target:    {TARGET_B}  |  Классов: {VOCAB_B}")
print(f"История:   {HIST_NORM_COLS_B}")
print(f"Данные:    Train={len(idx_tr_B)}, Val={len(idx_vl_B)}, Test={len(idx_te_B)}")
print(f"           Сумма={len(idx_tr_B)+len(idx_vl_B)+len(idx_te_B)} / исходный после фильтрации={len(df_B)}")
print(f"N_FOLDS={N_FOLDS}, seeds={MLP_KFOLD_SEEDS}, N_SEEDS_ABL={N_SEEDS_ABL}")
print(f"ABL_SEEDS == MLP_KFOLD_SEEDS: {ABL_SEEDS == MLP_KFOLD_SEEDS} (v14: единые сиды)")
print(f"LABEL_SMOOTHING={LABEL_SMOOTHING}")

print("\n--- АРХИТЕКТУРЫ (v14) ---")
print(f"  CareerMLP_Additive: retrain ablation ({N_SEEDS_ABL} seeds × {N_FOLDS} folds)")
print(f"  CareerMLP_Gated:    routing analysis only ({N_SEEDS_ABL} seeds × {N_FOLDS} folds)")
print(f"  Gate-веса = диагностика routing (не feature importance)")
print(f"  Temperature: init=log(1/sqrt({VOCAB_B})), clamp=[0.01, 1.0]")

print("\n--- BASELINES ---")
print("  Реализация бейзлайнов:")
print("  Majority  = Global Popularity Ranking (rank 1: most freq class, остальные по убыванию частоты)")
print("  Inertia   = Previous Role First (rank 1: job_2_position_norm, хвост: по частоте)")
print("  Bigram    = p(next|prev) с Лапласовским сглаживанием")
print(f"  Random (теор.) MRR ≈ {sum(1.0/k for k in range(1, VOCAB_B+1))/VOCAB_B*100:.2f}%")
for name, res in res_baselines.items():
    print(f"  {name:<14s}: {fmt(res)}")

print("\n--- КОМПОНЕНТЫ (val / test / gap) ---")
for name, sc_v, sc_t in [
    ("MLP KFold×seeds (add)",   sc_mlp_kfold_vl,  sc_mlp_kfold),
    ("MLP KFold×seeds (gated)", sc_mlp_gated_vl,  sc_mlp_gated),
]:
    vv = eval_full(sc_v, y_vl_B)["MRR"]*100
    tt = eval_full(sc_t, y_te_B)["MRR"]*100
    # v14: исправлена арифметика gap (замечание 41)
    gap = vv - tt
    print(f"  {name:<26s} val={vv:.2f}%  test={tt:.2f}%  gap={gap:.2f}pp")

print("\n  Примечание: val оценивается на отдельном сплите (до KFold);")
print("  финальная модель обучена на train+val с KFold и оценена на отдельном test.")
print("  Gap отражает дисперсию выборки, не overfitting.")

print("\n--- GATE ANALYSIS (gated MLP — diagnostic routing) ---")
print("  ⚠ Среднее gate ≠ feature importance (см. Шаг 12b)")
gate_mean_all = np.mean([g for g in all_gate_weights], axis=0)
gate_avg_all  = gate_mean_all.mean(axis=0)
gate_std_all  = gate_mean_all.std(axis=0)
for i, (bname, avg, std) in enumerate(zip(["career_emb", "skills_emb", "struct"],
                                           gate_avg_all, gate_std_all)):
    flag = ">" if avg > 1/3+0.05 else ("<" if avg < 1/3-0.05 else "≈")
    print(f"  g_{i+1} ({bname:<15s}): {avg:.3f} ± {std:.3f}  ({flag} 0.333 uniform)")

print("\n--- ENTROPY-STRATIFIED (v14, bootstrap CI + monotonicity test) ---")
print(df_ent.to_string())

pd.DataFrame({
    "metric":             ["MRR", "R@1", "R@3", "R@5"],
    "MLP_KFold_additive": [round(eval_full(sc_mlp_kfold, y_te_B)[k]*100, 2) for k in ["MRR","R@1","R@3","R@5"]],
    "MLP_KFold_gated":    [round(eval_full(sc_mlp_gated, y_te_B)[k]*100, 2) for k in ["MRR","R@1","R@3","R@5"]],
}).to_csv("results_taskB_v14.csv", index=False)

torch.save({
    "role_emb": role_emb, "role_to_id": role_to_id_B, "id_to_role": id_to_role_B,
    "ablation": ablation_results,
    "architecture": {
        "additive": "CareerMLP_Additive",
        "gated": "CareerMLP_Gated",
        "gate_avg": gate_avg_all.tolist(),
        "gate_std": gate_std_all.tolist(),
        "gate_disclaimer": "mean_gate != feature_importance",
    },
    "entropy": {
        "groups": {"Low": len(low_idx), "Mid": len(mid_idx), "High": len(high_idx)},
        "delta_by_group": delta_by_group,
        "bootstrap_n": N_BOOTSTRAP,
        "test": "bootstrap_p_value",
    },
    "best_cfg": best_cfg_name, "n_folds": N_FOLDS,
    "mlp_seeds": MLP_KFOLD_SEEDS, "abl_seeds": ABL_SEEDS,
    "target": TARGET_B,
    "history_norm": HIST_NORM_COLS_B, "history_desc": HIST_DESC_COLS_B,
    "version": "v14",
    "encoder_train_only": True,  # v14: R1+R2 only on idx_tr_B
}, "career_taskB_v14_final.pth")

print("\nСохранено: results_taskB_v14.csv, career_taskB_v14_final.pth")
print("         ablation_taskB_v14.csv, entropy_taskB_v14.csv")
print("         entropy_taskB_v14_ci.csv")


ИТОГИ — Career Path Prediction Task B v14 (IT domain, RU)
Энкодер:   intfloat/multilingual-e5-large (LAST стратегия, R1=3 эпохи + R2 hard negatives)
           R1+R2 обучены ТОЛЬКО на train (без val) — v14
Target:    job_1_position_norm  |  Классов: 35
История:   ['job_3_position_norm', 'job_2_position_norm']
Данные:    Train=45068, Val=5008, Test=8837
           Сумма=58913 / исходный после фильтрации=58965
N_FOLDS=5, seeds=[42, 123, 777], N_SEEDS_ABL=3
ABL_SEEDS == MLP_KFOLD_SEEDS: True (v14: единые сиды)
LABEL_SMOOTHING=0.1

--- АРХИТЕКТУРЫ (v14) ---
  CareerMLP_Additive: retrain ablation (3 seeds × 5 folds)
  CareerMLP_Gated:    routing analysis only (3 seeds × 5 folds)
  Gate-веса = диагностика routing (не feature importance)
  Temperature: init=log(1/sqrt(35)), clamp=[0.01, 1.0]

--- BASELINES ---
  Реализация бейзлайнов:
  Majority  = Global Popularity Ranking (rank 1: most freq class, остальные по убыванию частоты)
  Inertia   = Previous Role First (rank 1: job_2_position_norm,

In [19]:
# ══════════════════════════════════════════════════════════════════════════════
# ДОПОЛНИТЕЛЬНЫЙ АНАЛИЗ (добавить в notebook после Шага 13)
# Адресует замечания рецензии: утечка about, коллинеарность признаков,
# standalone предсказания, корректное описание бейзлайнов
# ══════════════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────────────
# ЯЧЕЙКА A: Проверка потенциальной утечки признака "О себе" (about)
# Методология: косинусное сходство between X_about_emb и role embeddings
# для примеров из test set. Если about-embedding уже "смотрит" на правильный
# target, это свидетельство утечки через декларацию намерений.
# ──────────────────────────────────────────────────────────────────────────────

print("=" * 70)
print("АНАЛИЗ A: Потенциальная утечка признака 'О себе' (about)")
print("=" * 70)

# Косинусное сходство X_about_emb с каждым из role embeddings
# (role_emb_np — матрица эмбеддингов ролей, уже есть в notebook)
X_about_te = feats["X_about_emb"][idx_te_B]  # (n_test, 1024)
X_about_te_norm = X_about_te / np.linalg.norm(X_about_te, axis=1, keepdims=True).clip(1e-8)
# role_emb_np уже нормализован
about_scores = X_about_te_norm @ role_emb_np.T  # (n_test, vocab)

res_about_only = eval_full(about_scores, y_te_B)
print(f"\n  About-only ранжирование (без обучения, только косинус):")
print(f"  {fmt(res_about_only)}")

# Для сравнения — случайный baseline (ожидаемый MRR = 1/VOCAB * ln(VOCAB) ≈ ?)
expected_random_mrr = sum(1.0/k for k in range(1, VOCAB_B+1)) / VOCAB_B
print(f"\n  Ожидаемый MRR случайного ранжирования: {expected_random_mrr*100:.2f}%")
print(f"  About-only MRR: {res_about_only['MRR']*100:.2f}%")

if res_about_only["MRR"] > expected_random_mrr * 2:
    print(f"\n  ⚠ About-embedding в {res_about_only['MRR']/expected_random_mrr:.1f}x лучше случайного.")
    print(f"    Это ожидаемо — about отражает профессиональные интересы.")
    print(f"    Для задачи Б (target = реальная последняя должность) это допустимо,")
    print(f"    но интерпретировать вклад about нужно с осторожностью:")
    print(f"    часть сигнала отражает намерения, часть — опыт.")
else:
    print(f"\n  ✓ About-embedding близок к случайному — прямой утечки нет.")

# Дополнительно: сравниваем вклад about в задаче Б относительно inertia
# Если about < inertia — он не является доминирующим источником утечки
inertia_mrr = res_baselines["Inertia"]["MRR"]
print(f"\n  Inertia baseline MRR: {inertia_mrr*100:.2f}%")
print(f"  About-only MRR:       {res_about_only['MRR']*100:.2f}%")
if res_about_only["MRR"] < inertia_mrr:
    print(f"  → About само по себе слабее Inertia: не является доминирующим предиктором.")
else:
    print(f"  → About само по себе сильнее Inertia: содержит сильный декларативный сигнал.")
    print(f"    Вклад about в ablation может частично отражать утечку (задача Б).")


# ──────────────────────────────────────────────────────────────────────────────
# ЯЧЕЙКА B: Standalone предсказания — skills-only и titles-only vs about-only
# Отвечает на замечание: нельзя сравнивать вклады без standalone baseline
# ──────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("АНАЛИЗ B: Standalone ранжирование без обучения (zero-shot косинус)")
print("Позволяет оценить raw signal каждого признака до MLP")
print("=" * 70)

standalone_results = {}

# Titles (X_seq) — карьерная история без описаний
X_seq_te = feats["X_seq"][idx_te_B]
X_seq_te_norm = X_seq_te / np.linalg.norm(X_seq_te, axis=1, keepdims=True).clip(1e-8)
sc_titles_zs = X_seq_te_norm @ role_emb_np.T
standalone_results["Titles only (zero-shot cosine)"] = eval_full(sc_titles_zs, y_te_B)

# Skills (X_skills_emb) — только навыки
X_skills_te = feats["X_skills_emb"][idx_te_B]
X_skills_te_norm = X_skills_te / np.linalg.norm(X_skills_te, axis=1, keepdims=True).clip(1e-8)
sc_skills_zs = X_skills_te_norm @ role_emb_np.T
standalone_results["Skills only (zero-shot cosine)"] = eval_full(sc_skills_zs, y_te_B)

# About (X_about_emb) — только текст "О себе"
standalone_results["About only (zero-shot cosine)"] = res_about_only

# Desc (X_doc) — история с описаниями
X_doc_te = feats["X_doc"][idx_te_B]
X_doc_te_norm = X_doc_te / np.linalg.norm(X_doc_te, axis=1, keepdims=True).clip(1e-8)
sc_doc_zs = X_doc_te_norm @ role_emb_np.T
standalone_results["History+desc (zero-shot cosine)"] = eval_full(sc_doc_zs, y_te_B)

print(f"\n  {'Признак':<40s}  {'MRR':>8s}  {'R@1':>8s}  {'R@3':>8s}  {'R@5':>8s}")
print(f"  {'-'*76}")
print(f"  {'Random (ожидаемое)':<40s}  {expected_random_mrr*100:7.2f}%  {'—':>8s}  {'—':>8s}  {'—':>8s}")
for name, res in standalone_results.items():
    print(f"  {name:<40s}  {res['MRR']*100:7.2f}%  {res['R@1']*100:7.2f}%  {res['R@3']*100:7.2f}%  {res['R@5']*100:7.2f}%")

print("""
  Интерпретация:
  - Zero-shot cosine НЕ является финальной моделью: MLP учится лучшим проекциям.
  - Значения выше случайного подтверждают, что каждый признак несёт сигнал
    о карьерной роли независимо.
  - Сравнение skills vs titles в zero-shot показывает, какой признак
    "ближе" к role embedding пространству по своей природе.
  - Это не заменяет ablation с обучением, но даёт интерпретируемую точку отсчёта.
""")


# ──────────────────────────────────────────────────────────────────────────────
# ЯЧЕЙКА C: Коллинеарность признаковых блоков
# Если два блока сильно скоррелированы → их нельзя сравнивать как независимые
# Метрика: средняя попарная косинусная близость между блоками
# ──────────────────────────────────────────────────────────────────────────────

print("=" * 70)
print("АНАЛИЗ C: Коллинеарность признаковых блоков (cosine similarity)")
print("Контекст для интерпретации ablation: независимы ли блоки?")
print("=" * 70)

# Вычисляем попарную cosine similarity между блоками на test set
# Используем mean(diag(A @ B.T)) — среднее попарное сходство
blocks = {
    "Titles (X_seq)":   feats["X_seq"][idx_te_B],
    "Desc (X_doc)":     feats["X_doc"][idx_te_B],
    "Skills":           feats["X_skills_emb"][idx_te_B],
    "About":            feats["X_about_emb"][idx_te_B],
}

# Нормализуем каждый блок
blocks_norm = {
    name: arr / np.linalg.norm(arr, axis=1, keepdims=True).clip(1e-8)
    for name, arr in blocks.items()
}

block_names = list(blocks_norm.keys())
print(f"\n  Средняя попарная косинусная близость между блоками (test set):")
print(f"  (1.0 = идентичны, 0.0 = ортогональны)\n")

header = f"  {'':22s}" + "".join(f"  {n[:12]:>12s}" for n in block_names)
print(header)
print("  " + "-" * (22 + 14 * len(block_names)))

similarity_matrix = {}
for n1 in block_names:
    row_str = f"  {n1[:22]:<22s}"
    similarity_matrix[n1] = {}
    for n2 in block_names:
        # Mean cosine similarity: mean of element-wise dot products of normalized vectors
        mean_cos = float(np.mean(np.sum(blocks_norm[n1] * blocks_norm[n2], axis=1)))
        similarity_matrix[n1][n2] = mean_cos
        row_str += f"  {mean_cos:12.3f}"
    print(row_str)

print("""
  Интерпретация:
  - Titles vs Desc: ожидаемо высокая схожесть (desc включает title)
  - Skills vs остальные: умеренная → skills несут частично независимую информацию
  - About vs остальные: если высокая → about дублирует сигнал истории/навыков
  
  ВАЖНО для интерпретации ablation:
  Если два блока сильно коррелированы (>0.7), маржинальный прирост
  от добавления второго блока будет занижен — часть сигнала уже учтена.
  Это НЕ означает, что второй блок бесполезен — он может быть важен
  при отсутствии первого.
""")


# ──────────────────────────────────────────────────────────────────────────────
# ЯЧЕЙКА D: Корректное переописание бейзлайнов
# Majority и Inertia в коде используют popularity ranking, а не single label —
# это нужно явно задокументировать чтобы метрики были интерпретируемы
# ──────────────────────────────────────────────────────────────────────────────

print("=" * 70)
print("АНАЛИЗ D: Документация бейзлайнов")
print("=" * 70)

print(f"""
  Бейзлайны в данной реализации:

  Majority (Global Popularity Ranking):
    - rank 1: наиболее частый класс в train ('{id_to_role_B[major_cls]}')
    - ranks 2..{VOCAB_B}: остальные классы по убыванию частоты
    - Это popularity ranking, а не single-label prediction
    - R@1 ≠ R@5 ≠ R@k (корректно)

  Inertia (Previous Role First):
    - rank 1: предыдущая должность (job_2_position_norm) если известна,
              иначе — наиболее частый класс
    - ranks 2..{VOCAB_B}: остальные классы по убыванию частоты (fallback = majority)
    - R@1 = доля случаев, где job_2 == job_1 (инерция)
    - R@5 > R@1 потому что tail ranks заполнены популярными классами

  Bigram Prior:
    - rank 1..{VOCAB_B}: p(next_role | prev_role) по train, Лапласовское сглаживание
    - Если prev_role не встречалась в train: равномерное распределение

  Случайное ранжирование (теоретическое):
    - Ожидаемый MRR = {expected_random_mrr*100:.2f}%
    - Ожидаемый R@1 = {100.0/VOCAB_B:.2f}%
    - Ожидаемый R@5 = {500.0/VOCAB_B:.2f}%
""")

# Проверяем арифметически что R@1 <= R@5 для всех бейзлайнов
print("  Проверка монотонности R@1 ≤ R@3 ≤ R@5 для бейзлайнов:")
for name, res in res_baselines.items():
    ok = res["R@1"] <= res["R@3"] <= res["R@5"]
    print(f"    {name:<14s}: R@1={res['R@1']*100:.2f}% ≤ R@3={res['R@3']*100:.2f}% ≤ R@5={res['R@5']*100:.2f}%  {'✓' if ok else '✗'}")


# ──────────────────────────────────────────────────────────────────────────────
# ЯЧЕЙКА E: Проверка арифметики сплита
# ──────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("АНАЛИЗ E: Верификация разбиения данных")
print("=" * 70)

total_check = len(idx_tr_B) + len(idx_vl_B) + len(idx_te_B)
print(f"\n  Train:     {len(idx_tr_B):>7d}")
print(f"  Val:       {len(idx_vl_B):>7d}")
print(f"  Test:      {len(idx_te_B):>7d}")
print(f"  ──────────────────")
print(f"  Сумма:     {total_check:>7d}")
print(f"  Исходный:  {len(df_B):>7d}")
diff = len(df_B) - total_check
if diff == 0:
    print(f"  ✓ Сумма совпадает — утечек в разбиении нет")
else:
    print(f"  ⚠ Расхождение: {diff} записей отброшено при keep_B() фильтрации")
    print(f"    (записи с target не из финального vocab — корректно)")

# Проверка пересечений train/val/test
tr_set = set(idx_tr_B.tolist())
vl_set = set(idx_vl_B.tolist())
te_set = set(idx_te_B.tolist())
print(f"\n  Пересечений train∩val:  {len(tr_set & vl_set)}")
print(f"  Пересечений train∩test: {len(tr_set & te_set)}")
print(f"  Пересечений val∩test:   {len(vl_set & te_set)}")
if not (tr_set & vl_set) and not (tr_set & te_set) and not (vl_set & te_set):
    print(f"  ✓ Разбиение без пересечений")
else:
    print(f"  ✗ ОШИБКА: есть пересечения между сплитами!")

# Val/test gap
# Примечание: val оценивается на отдельном сплите до KFold;
# финальная модель обучается на train+val с KFold
# gap = разность val_MRR (по отдельному val split) и test_MRR (по test)
# Это НЕ overfitting — это разный набор данных
print(f"\n  Примечание о val-test gap:")
print(f"  val_MRR и test_MRR измерены на разных подвыборках одного распределения.")
print(f"  Финальная модель (KFold) обучена на train+val и оценена на отдельном test.")
print(f"  Gap отражает: (1) дисперсию выборки, (2) эффект использования val в обучении.")
print(f"  Он НЕ является свидетельством overfitting при условии отсутствия")
print(f"  data leakage через test set в процессе выбора гиперпараметров.")

АНАЛИЗ A: Потенциальная утечка признака 'О себе' (about)

  About-only ранжирование (без обучения, только косинус):
  MRR=36.12%  R@1=20.82%  R@3=40.56%  R@5=50.38%

  Ожидаемый MRR случайного ранжирования: 11.85%
  About-only MRR: 36.12%

  ⚠ About-embedding в 3.0x лучше случайного.
    Это ожидаемо — about отражает профессиональные интересы.
    Для задачи Б (target = реальная последняя должность) это допустимо,
    но интерпретировать вклад about нужно с осторожностью:
    часть сигнала отражает намерения, часть — опыт.

  Inertia baseline MRR: 49.75%
  About-only MRR:       36.12%
  → About само по себе слабее Inertia: не является доминирующим предиктором.

АНАЛИЗ B: Standalone ранжирование без обучения (zero-shot косинус)
Позволяет оценить raw signal каждого признака до MLP

  Признак                                        MRR       R@1       R@3       R@5
  ----------------------------------------------------------------------------
  Random (ожидаемое)                          1

In [20]:
# ══════════════════════════════════════════════════════════════════════════════
# АНАЛИЗ F: Проверка гипотезы об артефакте частотности в энтропийной стратификации
#
# Замечание рецензии: группа High может быть просто "редкие роли", а не
# "объективно непредсказуемые переходы". Если это так — весь эффект
# монотонного роста ΔMRR объясняется частотностью, а не энтропией.
#
# Что проверяем:
# 1. Корреляция между H(l) и support(l) — насколько они связаны
# 2. Метрики отдельно по частотности (без энтропии) — воспроизводится ли эффект
# 3. Частичная корреляция: есть ли эффект энтропии после контроля за частотностью
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("АНАЛИЗ F: Энтропия vs частотность — артефакт или реальный эффект?")
print("=" * 70)

import numpy as np
from collections import Counter
from scipy.stats import spearmanr, pearsonr

# ── Шаг 1: Восстанавливаем support и entropy для каждой предыдущей роли ───────
# JOB2_COL_B — ближайшая предыдущая роль (уже определена в notebook)
# idx_tr_B, y_B, role_to_id_B — уже определены

# Считаем по train: для каждой prev_role → распределение next_role
from collections import defaultdict

prev_to_next_train = defaultdict(list)
for i in idx_tr_B:
    prev = str(df_B.iloc[i].get(JOB2_COL_B, "") or "").strip()
    target = y_B[i]
    if prev not in SKIP and target in role_to_id_B:
        prev_to_next_train[prev].append(role_to_id_B[target])

# Для каждой prev_role: support (число наблюдений) и энтропия
role_support = {}
role_entropy  = {}

for prev, targets in prev_to_next_train.items():
    n = len(targets)
    role_support[prev] = n
    cnt = Counter(targets)
    probs = np.array([cnt[t] / n for t in cnt])
    # Лапласовское сглаживание (как в bigram prior)
    cnt_smoothed = np.ones(VOCAB_B)
    for t, c in cnt.items():
        cnt_smoothed[t] += c
    probs_smoothed = cnt_smoothed / cnt_smoothed.sum()
    role_entropy[prev] = float(-np.sum(probs_smoothed * np.log(probs_smoothed + 1e-12)))

# ── Шаг 2: Корреляция H(l) и support(l) ──────────────────────────────────────
print("\n--- Шаг 1: Корреляция entropy ↔ support ---")

common_roles = [r for r in role_entropy if r in role_support]
supports = np.array([role_support[r] for r in common_roles])
entropies = np.array([role_entropy[r] for r in common_roles])

rho, p_rho = spearmanr(supports, entropies)
r, p_r     = pearsonr(np.log1p(supports), entropies)

print(f"\n  Число ролей с известным support: {len(common_roles)}")
print(f"  Spearman ρ(support, entropy):     {rho:+.3f}  (p={p_rho:.4f})")
print(f"  Pearson r(log(support), entropy): {r:+.3f}  (p={p_r:.4f})")

if abs(rho) > 0.5:
    print(f"\n  ⚠ Сильная корреляция между support и entropy.")
    print(f"    Это означает, что энтропийные группы смешивают два эффекта:")
    print(f"    (1) реальную непредсказуемость перехода")
    print(f"    (2) шум оценки из-за малого числа наблюдений")
elif abs(rho) > 0.3:
    print(f"\n  ⚡ Умеренная корреляция. Частотность частично объясняет энтропию,")
    print(f"     но не полностью. Эффект энтропии не является чистым артефактом.")
else:
    print(f"\n  ✓ Слабая корреляция. Энтропия и частотность — во многом независимые")
    print(f"    характеристики. Эффект энтропии не объясняется частотностью.")

# ── Шаг 3: Назначаем каждому тестовому примеру support его prev_role ──────────
print("\n--- Шаг 2: Support в энтропийных группах ---")

# Восстанавливаем энтропийные группы из notebook
# (low_idx, mid_idx, high_idx должны быть определены в notebook)
# Если нет — пересчитываем:

test_prev_roles = [
    str(df_B.iloc[i].get(JOB2_COL_B, "") or "").strip()
    for i in idx_te_B
]
test_entropies = np.array([
    role_entropy.get(r, np.nan) for r in test_prev_roles
])
test_supports  = np.array([
    role_support.get(r, np.nan) for r in test_prev_roles
])

# Квантили энтропии (как в notebook)
valid_mask = ~np.isnan(test_entropies)
ents_valid = test_entropies[valid_mask]
q33 = np.nanpercentile(test_entropies, 33)
q66 = np.nanpercentile(test_entropies, 66)

low_idx_f  = np.where(test_entropies <= q33)[0]
mid_idx_f  = np.where((test_entropies > q33) & (test_entropies <= q66))[0]
high_idx_f = np.where(test_entropies > q66)[0]

print(f"\n  Энтропийные группы (test set):")
print(f"  {'Группа':6s}  {'n':>5s}  {'median H':>10s}  {'median support':>15s}  {'mean support':>12s}")
print(f"  {'-'*55}")
for gname, gidx in [("Low", low_idx_f), ("Mid", mid_idx_f), ("High", high_idx_f)]:
    g_ent = test_entropies[gidx]
    g_sup = test_supports[gidx]
    valid_sup = g_sup[~np.isnan(g_sup)]
    print(f"  {gname:6s}  {len(gidx):5d}  {np.nanmedian(g_ent):10.3f}  "
          f"{np.nanmedian(valid_sup):15.1f}  {np.nanmean(valid_sup):12.1f}")

# ── Шаг 4: Стратификация по ЧАСТОТНОСТИ (без энтропии) ───────────────────────
# Ключевой тест: если разбить по support вместо entropy —
# воспроизведётся ли тот же паттерн ΔMRR?
print("\n--- Шаг 3: Стратификация по ЧАСТОТНОСТИ (контрольный эксперимент) ---")
print("  Если ΔMRR монотонно растёт по частотности так же как по энтропии →")
print("  эффект объясняется частотностью, а не энтропией.\n")

# Квантили support
q33_sup = np.nanpercentile(test_supports, 33)
q66_sup = np.nanpercentile(test_supports, 66)

low_sup_idx  = np.where(test_supports <= q33_sup)[0]
mid_sup_idx  = np.where((test_supports > q33_sup) & (test_supports <= q66_sup))[0]
high_sup_idx = np.where(test_supports > q66_sup)[0]

print(f"  Support квантили: Q33={q33_sup:.0f}, Q66={q66_sup:.0f}")
print(f"  Low (редкие):  n={len(low_sup_idx)}, median_support={np.nanmedian(test_supports[low_sup_idx]):.1f}")
print(f"  Mid:           n={len(mid_sup_idx)}, median_support={np.nanmedian(test_supports[mid_sup_idx]):.1f}")
print(f"  High (частые): n={len(high_sup_idx)}, median_support={np.nanmedian(test_supports[high_sup_idx]):.1f}")

MIN_GROUP_SIZE = 30  # минимум для bootstrap

print(f"\n  ΔMRR(+desc − base) по группам ЧАСТОТНОСТИ:")
print(f"  {'Группа':6s}  {'n':>5s}  {'ΔMRR base→desc':>16s}  {'base MRR':>10s}  {'desc MRR':>10s}")
print(f"  {'-'*55}")

delta_by_support = {}
for gname, gidx in [("Low\n(редкие)", low_sup_idx),
                    ("Mid",            mid_sup_idx),
                    ("High\n(частые)", high_sup_idx)]:
    label = gname.replace("\n", " ")
    if len(gidx) < MIN_GROUP_SIZE:
        print(f"  {label:12s}  {len(gidx):5d}  — слишком мала группа —")
        continue
    mrr_base = eval_full(sc_base_te[gidx], y_te_B[gidx])["MRR"] * 100
    mrr_desc = eval_full(sc_desc_te[gidx], y_te_B[gidx])["MRR"] * 100
    delta = mrr_desc - mrr_base
    delta_by_support[label] = delta
    print(f"  {label:12s}  {len(gidx):5d}  {delta:+14.2f}pp  {mrr_base:9.2f}%  {mrr_desc:9.2f}%")

# ── Шаг 5: Сравнение паттернов entropy vs frequency ──────────────────────────
print("\n--- Шаг 4: Сравнение паттернов ---")
print(f"\n  ΔMRR по энтропийным группам (из notebook):")
print(f"  Low  → Mid  → High: ожидается рост (гипотеза H2)")
print(f"\n  ΔMRR по группам частотности:")
print(f"  Low (редкие) → Mid → High (частые): ожидается ПАДЕНИЕ если частотность важна")
print(f"  (редкие роли → меньше данных → сложнее предсказывать → больший прирост от доп. признаков)")

print(f"""
  Интерпретация результатов:

  Если ΔMRR по частотности:
    РАСТЁТ от High(частые) к Low(редкие):
      → Эффект частично объясняется частотностью.
        Надо добавить оговорку: "эффект энтропии конфаундируется частотностью".
    
    НЕ МОНОТОНЕН или слабее чем по энтропии:
      → Энтропия несёт самостоятельную объяснительную силу
        поверх частотности. Гипотеза H2 более устойчива.

  Ключевое сравнение: если High-entropy группа содержит ОДНОВРЕМЕННО
  редкие И частые роли → энтропия не сводится к частотности.
""")

# ── Шаг 6: Роли в High-entropy группе с их support ───────────────────────────
print("--- Шаг 5: Состав High-entropy группы ---")
print("  Есть ли в High-entropy группе роли с высоким support?\n")

# Собираем уникальные prev_roles в High-entropy test group
high_prev_roles = [test_prev_roles[i] for i in high_idx_f]
high_role_stats = {}
for r in set(high_prev_roles):
    if r in role_entropy:
        high_role_stats[r] = {
            "entropy": role_entropy[r],
            "support": role_support.get(r, 0),
            "n_test":  high_prev_roles.count(r)
        }

# Сортируем по support убыванию
sorted_high = sorted(high_role_stats.items(), key=lambda x: -x[1]["support"])

print(f"  {'Роль':<35s}  {'support':>8s}  {'entropy':>8s}  {'n_test':>7s}")
print(f"  {'-'*65}")
for role, stats in sorted_high[:15]:  # топ-15
    print(f"  {role:<35s}  {stats['support']:8d}  {stats['entropy']:8.3f}  {stats['n_test']:7d}")

# Проверяем есть ли высокочастотные роли в High-entropy
high_supports_in_high_ent = [s["support"] for s in high_role_stats.values()]
median_overall = np.median(list(role_support.values()))
roles_above_median = sum(1 for s in high_supports_in_high_ent if s > median_overall)
print(f"\n  Медианный support по всем ролям: {median_overall:.0f}")
print(f"  Ролей в High-entropy с support > медианы: {roles_above_median}/{len(high_role_stats)}")

if roles_above_median > len(high_role_stats) * 0.3:
    print(f"\n  ✓ В High-entropy группе есть роли с высоким support.")
    print(f"    Это означает, что высокая энтропия не сводится к редкости роли.")
    print(f"    Некоторые частые роли действительно имеют разнообразные переходы.")
else:
    print(f"\n  ⚠ В High-entropy группе преобладают редкие роли.")
    print(f"    Артефакт частотности частично объясняет эффект. Добавить оговорку.")

print("""
  Вывод для текста ВКР:
  Добавить в раздел ограничений:
  "Энтропия H(l) оценивалась по обучающей выборке, что означает
  частичную конфаундированность с частотностью роли: редко встречающиеся
  роли имеют менее стабильные оценки распределения переходов. Для проверки
  была проведена параллельная стратификация по частотности; [сравнить
  результаты]. Наличие в High-entropy группе ролей с высоким support
  [подтверждает/ставит под сомнение] самостоятельный вклад энтропии
  как характеристики сложности перехода."
""")

АНАЛИЗ F: Энтропия vs частотность — артефакт или реальный эффект?

--- Шаг 1: Корреляция entropy ↔ support ---

  Число ролей с известным support: 50
  Spearman ρ(support, entropy):     -0.905  (p=0.0000)
  Pearson r(log(support), entropy): -0.900  (p=0.0000)

  ⚠ Сильная корреляция между support и entropy.
    Это означает, что энтропийные группы смешивают два эффекта:
    (1) реальную непредсказуемость перехода
    (2) шум оценки из-за малого числа наблюдений

--- Шаг 2: Support в энтропийных группах ---

  Энтропийные группы (test set):
  Группа      n    median H   median support  mean support
  -------------------------------------------------------
  Low      3175       1.812           3076.0        5017.7
  Mid      2947       2.065           5327.0        4521.7
  High     1756       2.329            627.0        1083.7

--- Шаг 3: Стратификация по ЧАСТОТНОСТИ (контрольный эксперимент) ---
  Если ΔMRR монотонно растёт по частотности так же как по энтропии →
  эффект объясняется